# Proyecto Prisma — Tópicos, sentimiento y ABSA sobre preguntas abiertas

**Proyecto:** Capstone NLP — MINE009 · **Módulo:** `src/nlp_utilidades/modulos_nlp.py` (versión ft-r4.15) · **Encuesta:** se elige en la celda CONFIG (`CLAVE_FOCAL`): obs19, obs20, obs23, obs24, obs25 o `acred_sugerencias_complementarias`

---

## Qué es una cláusula

Una respuesta abierta suele valorar varias cosas a la vez:

> *"Explica muy bien los temas, **pero** nunca responde los correos."*

Contiene **dos unidades de opinión**: una positiva sobre la explicación y una negativa sobre la
comunicación. Cada unidad es una **cláusula**. Se obtienen con un analizador sintáctico del español
(spaCy `es_core_news_lg`) que separa las proposiciones de la respuesta sin partir lo que va junto
(perífrasis como *tiene que*, subordinadas sin verbo conjugado, marcadores de lista).

La cláusula es la **unidad de análisis de todo el pipeline**: se le asigna un aspecto (tópico) y una
polaridad, y la matriz ABSA cruza exactamente esas dos asignaciones.

## Decisiones de diseño

| # | Problema | Solución en este pipeline |
|---|---|---|
| H1 | Archivos de salida de corridas distintas mezclados | Una carpeta por corrida con huella; la exportación relee y verifica |
| H2 | Tópicos por documento y sentimiento por cláusula | Tópicos modelados directamente sobre cláusulas |
| H3 | Tópicos de tono, de género ("ella") y de tratamiento ("doctor") | **Texto de aspecto**: sin valorativos, persona, negadores ni nombres (`neutralizacion.xlsx`) |
| H4 | "Sin comentarios" dentro de tópicos | Filtro de **no opinión** antes de modelar |
| H5 | Etiquetas con palabras vacías | Stopwords NLTK + globales + particulares por encuesta (`stopwords.xlsx`) |
| H6 | El puntaje premiaba la concentración | Puertas duras + Pareto + estabilidad (ARI); si nada cumple, falla explícito |
| H7 | Reasignación distinta en la búsqueda y en el modelo final | Una sola función, verificada por huella de partición |
| H8 | NEU como zona de duda, sin validación | Probabilidades, polaridad continua, validación contra la nota y contra etiquetas humanas |
| H9 | Prioridad = conteo de negativos | Límite inferior de Wilson × log(volumen), con n mínimo |
| H10 | Abreviaturas partidas ("Dr." ) | Protección de abreviaturas, decimales e iniciales |
| H13 | F1 medido en una muestra no representativa | F1 **reponderado a la población** |
| H14 | Texto escrito por el docente validado contra la nota de los estudiantes | `autor_texto`: esa validación queda NO_APLICA |
| H15 | RoBERTuito subestima elogios y lee sugerencias como neutras | **Calibrador** opcional, aprendido de anotaciones humanas y medido con validación cruzada |
| H17 | Un tópico "cajón" de palabras sueltas encabezaba la priorización | `confiable` = no (cohesión baja y ≥ 2× palabras sueltas del corpus): baja al final del ranking |
| H20 | La cobertura mezclaba asignación directa y por cercanía | Columna `asignacion` (directa / reasignada / sin_topico) y métrica `cobertura_directa` |
| H23 | Se podía elegir una configuración inestable | Se elige la primera configuración que cumple **y** es estable (ARI ≥ 0,70) |
| H24 | Tópicos redundantes tras una sola pasada de fusión | La fusión se repite hasta que ningún par supere `umbral_fusion` |
| H25 | Módulo viejo en memoria sin detectarse | `HUELLA_ARCHIVO`: el Paso 0 compara la copia en memoria con la del disco |
| H27 | Un solo calibrador para todos los tipos de pregunta | Modelo propio por tipo de pregunta solo si en validación cruzada no rinde peor |
| H28 | En fortalezas, el calibrador empeoraba frente al juicio humano | Interruptor `usar_calibrador` (`PERFILES`, Paso 0a), decidido por encuesta con validación humana |
| H30 | UMAP/HDBSCAN agrupaba por similitud, sin garantía de que los grupos fueran los aspectos del equipo | **Modo `u4`**: un clasificador por pregunta (mpnet con las últimas 4 capas descongeladas + cabeza lineal) entrenado **solo con etiquetas auditadas a mano** decide el aspecto. UMAP/HDBSCAN queda para el modo `bertopic` y, en `u4`, para descubrir aspectos nuevos en el **residual** |


## Cómo usar este cuaderno (léalo primero)

**Modo de aspectos:** `modo_aspectos` en `PERFILES` (`"u4"` por defecto; `"bertopic"` reproduce las corridas de ft-r4.14). U4 exige un **archivo de etiquetas auditado por pregunta** en `RAIZ/data/etiquetas_manuales/` (archivo fijado por pregunta en `PERFILES`) (ver Paso U) y GPU para entrenar; con `SOLO_ETIQUETAR = True` se etiqueta con el modelo ya guardado, sin reentrenar.

**Parámetros por encuesta:** todos en la celda **CONFIG** (Paso 0a, diccionario `PERFILES`). Para cambiar de encuesta solo se edita `CLAVE_FOCAL`. Léxicos y catálogo: `RAIZ/data/utilities`. Todas las rutas se definen en un solo bloque, `RUTAS` (Paso 0, celda de entorno).

**Sesión A — corrida completa (con GPU):** `Entorno de ejecución → Cambiar tipo → T4 GPU` y
`Ejecutar todo`.

| Pasos | Qué produce |
|---|---|
| 0–4 | Entorno, léxicos, corpus y segmentación en cláusulas |
| U1–U7 | **Modo `u4` (por defecto):** entrenar/reutilizar U4, etiquetar el corpus, resultados, residual (UMAP/HDBSCAN solo sobre lo que U4 no reconoce) y modelo final |
| 5–8b | Solo en modo `bertopic` (encoder, grid, modelo final, aspectos, 8b): se omiten solos en modo `u4` |
| 9–12 | Sentimiento, validación, ABSA, scorecard y exportación (ambos modos) |
| 12b | **Modo `u4`:** las 4 tablas del contrato de datos v0.2 en `data/processed/nlp/<id_corrida_nlp>/` y el modelo UMAP/HDBSCAN del residual |
| 13 | Opcional: recalificar con la auditoría humana diligenciada |
| 14 | Opcional (`CORRER_RESTO`): otras encuestas de `MIS_ENCUESTAS`, rejilla `rapida` |
| 15 | `outputs_full.zip` descargado a su computador |

Con `GUARDAR_EN_DRIVE = True` los resultados quedan además en Drive.

**Entre sesiones — diligenciar (sin Colab):** abra `auditoria.xlsx` de la corrida (hojas
`muestra_pureza` y `muestra_manual`). **No toque la hoja `corrida`**: identifica la corrida.

**Sesión B — recalificar (minutos, sin GPU):** ejecute **solo el Paso 0 y el Paso 16**.


## Paso 0a — Configuración (celda CONFIG)

Es la **única celda que se edita** para cambiar de encuesta: `CLAVE_FOCAL`. Los parámetros de cada
encuesta viven en el diccionario `PERFILES` (calibrador, criterios de negocio, política de fusión,
semillas de ARI, auditoría y, desde ft-r4.15, `modo_aspectos`, el bloque `aspectos_u4` —archivo de etiquetas auditado,
hoja, columnas, hiperparámetros— y el bloque `residual_hdbscan`). La celda es Python puro (no importa el módulo), así que puede ejecutarse sola.

Los **léxicos** (`stopwords`, `negaciones`, `neutralizacion`, `taxonomia`) son un
único juego en `RAIZ/data/utilities`; no hay un juego por encuesta. (`accents.xlsx` ya no se usa: se normaliza con `unicodedata`.)

**Interruptores del modo u4:** `REUTILIZAR_MODELO` (no reentrena si ya hay un modelo con la misma huella de datos + configuración), `PERMITIR_CPU` (entrenar exige GPU) y `SOLO_ETIQUETAR` (carga el modelo guardado y etiqueta, sin reentrenar). Los umbrales `umbral_comentario_general` y `residual_hdbscan['umbral']` (0,50) son **provisionales**: se ajustan con los diagnósticos de los Pasos U2 y U4.


In [1]:
import sys
if "google.colab" in sys.modules:   # único montaje de Drive (la celda de RUTAS ya no monta)
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount("/content/drive")


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ══ CONFIG ÚNICA ═══════════════════════════════════════════════════════════════
# Para cambiar de encuesta se edita SOLO la línea CLAVE_FOCAL. Todo lo demás sale de PERFILES.
CLAVE_FOCAL = "obs19_aspectos_resaltar"
#   obs19_aspectos_resaltar | obs20_oportunidades_mejora | obs23_percepcion_progreso
#   obs24_aspectos_resaltar | obs25_oportunidades_mejora | acred_sugerencias_complementarias

# Carpeta del proyecto en Drive (solo Colab). Local: RAIZ se detecta por pyproject.toml.
RAIZ_COLAB = "/content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/capstonenlpencuestas_mine9"

# ── Interruptores del modo u4 (ver Paso U) ─────────────────────────────────────
REUTILIZAR_MODELO = True   # si ya existe un modelo U4 con la misma huella de datos + configuración, NO se reentrena
PERMITIR_CPU = False       # entrenar U4 exige GPU (T4). True solo para pruebas de humo pequeñas (en CPU tarda horas)
SOLO_ETIQUETAR = False     # True = no entrena: carga el modelo guardado (ultimo.json de la pregunta) y etiqueta el corpus

# ── Salidas del contrato v0.2 (Paso 12b): decisiones abiertas del equipo (spec §5), con el valor de hoy ──
SALIDAS_CONTRATO = dict(
    regla_opinion_general="palabras",   # "palabras" (hoy: respuesta de <= 5 palabras con baja confianza) | "contenido" (pendiente)
    periodo_emergentes=None,             # None = periodo más reciente de la pregunta (EMERGENTE_<periodo>_<n>)
)

# Valores de base. Los que coinciden con los defaults del módulo se escriben explícitos para
# que el perfil sea legible sin abrir modulos_nlp.py.
_PRISMA = dict(
    usar_calibrador=False,               # interruptor por encuesta (H28)
    n_topicos_max=35,                    # CriteriosNegocio (modo bertopic)
    pct_outliers_max=30.0,               # CriteriosNegocio (modo bertopic)
    peso_silueta=0.35,                   # CriteriosNegocio (default del módulo)
    peso_estabilidad=0.20,               # CriteriosNegocio (default del módulo)
    rejilla="completa",                  # corrida de entrega (modo bertopic); "rapida" sólo para pruebas
    cluster_selection_method="leaf",     # Config (default del módulo)
    politica_fusion="original",          # "original" | "raiz_coseno"
    ari_semillas=(7, 2024),              # Config (default del módulo)
    n_finalistas_estabilidad=3,          # Config (default del módulo)
    permitir_no_cumple=True,             # si nada cumple, se usa la fila más cercana y el scorecard lo registra
    auditoria_diligenciada=False,        # cambie a True cuando haya llenado el Excel de auditoría
    ruta_muestra_anterior=None,          # opcional: auditoria.xlsx de otra corrida con muestra_manual ya diligenciada

    # ── Aspectos supervisados U4 (ft-r4.15) ──
    modo_aspectos="u4",                  # "u4" (U4 decide el aspecto) | "bertopic" (UMAP/HDBSCAN, comportamiento de ft-r4.14)
    semilla_u4=42,                       # semilla de TODO el azar del entrenamiento (cabeza, dropout, lotes); 42 = la del experimento
    max_palabras_respuesta_corta=5,      # respuesta corta = respuesta completa (sin limpiar) con <= N palabras (split())
    umbral_comentario_general=0.80,      # PROVISIONAL: respuesta corta con prob. máx. de U4 < umbral -> «Comentario General»
                                         #   (se ajusta con el diagnóstico del Paso U2: % que pasaría a CG con 0.3 … 0.7)
    aspectos_u4=dict(
        activo=True,                     # False = esta pregunta se procesa en modo bertopic
        encuesta_etiquetas="obs19",      # nombre en la hoja «aspectos» del Excel de validación y en el nombre del archivo auditado
        # Archivo de etiquetas: ruta relativa a RAIZ (params_u4() la vuelve absoluta); cada perfil fija el suyo.
        # None = se busca el patrón --Prisma_Validacion_Humana-<encuesta_etiquetas>*auditado*.xlsx en <carpeta_etiquetas>.
        archivo_etiquetas=None,
        carpeta_etiquetas="data/etiquetas_manuales",
        hoja_etiquetas="obs19",
        col_fragmento="fragmento_final_entrenamiento",
        col_aspecto="aspecto_final_entrenamiento",
        col_respuesta="respuesta_completa",   # obs19 es la excepción: "respuesta_completa_original" (ver su perfil)
        # El vocabulario de aspectos sale SOLO del archivo auditado (mayúsculas/tildes normalizadas; ya no se lee la hoja «aspectos»).
        hiperparametros=dict(            # U4 (los del experimento de regresión logística)
            k=4, lr_enc=2e-5, lr_cabeza=1e-3, max_length=64, batch=32, epocas_max=10, paciencia=3,
            weight_decay=0.01, warmup=0.10, label_smoothing=0.1, dropout_cabeza=0.1, fp16=True,
            n_interno=8, frac_holdout=0.15,
        ),
    ),
    residual_hdbscan=dict(               # descubrimiento sobre lo que U4 no reconoce (cláusulas largas de baja probabilidad)
        activo=True,                     # False = sin residual: toda cláusula ASPECTO conserva el aspecto de U4
        umbral=0.10,                     # prob. máx. de U4 < umbral -> residual (PROVISIONAL; distinto de umbral_comentario_general)
        min_palabras=6,                  # la cláusula debe tener >= 6 palabras (> 5) para ser residual
        minimo_residual=None,            # None = MINIMO_MODELABLES del módulo (300): con menos no se ejecuta HDBSCAN
        rejilla="rapida",                # rejilla del modelado del residual
        criterios=dict(                  # criterios de negocio RELAJADOS (el residual es pequeño y heterogéneo)
            n_topicos_min=2, n_topicos_max=12, pct_outliers_max=40.0,
            pct_topico_mayor_max=60.0, pct_top2_max=85.0, silueta_min=0.0, ari_min=0.50,
        ),
    ),
)

PERFILES = {
    # Evaluación docente: sin calibrador (la validación humana mostró que empeora, H28).
    "obs19_aspectos_resaltar": {
        **_PRISMA, "auditoria_diligenciada": True,
        "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "obs19", "hoja_etiquetas": "obs19",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-obs19 - auditado-27102026.xlsx",
                        "col_respuesta": "respuesta_completa_original"},
    },
    # Con calibrador: decisión preliminar, pendiente de su validación humana.
    "obs20_oportunidades_mejora": {
        **_PRISMA, "usar_calibrador": True,
        "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "obs20", "hoja_etiquetas": "obs20",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-obs20 - auditado-10112026.xlsx",
                        "col_respuesta": "respuesta_completa"},
    },
    # Autoevaluación docente: calibrador PENDIENTE de decidir con su validación humana (H28).
    # obs23–obs25: archivo explícito en data/etiquetas_manuales/ (no llevan «auditado» en el nombre); hoja = nombre de la encuesta;
    # su columna de respuesta es "respuesta_completa" (solo obs19 usa "_original").
    "obs23_percepcion_progreso": {**_PRISMA, "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "obs23", "hoja_etiquetas": "obs23",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-obs23-13112026.xlsx", "col_respuesta": "respuesta_completa"}},
    "obs24_aspectos_resaltar": {**_PRISMA, "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "obs24", "hoja_etiquetas": "obs24",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-obs24-13112026.xlsx", "col_respuesta": "respuesta_completa"}},
    "obs25_oportunidades_mejora": {**_PRISMA, "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "obs25", "hoja_etiquetas": "obs25",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-obs25-13112026.xlsx", "col_respuesta": "respuesta_completa"}},
    # Acreditación (corpus pequeño): menos tópicos, fusión por raíz+coseno, 4 semillas de ARI (modo bertopic).
    "acred_sugerencias_complementarias": {
        **_PRISMA,
        "n_topicos_max": 15,
        "peso_silueta": 0.25,
        "peso_estabilidad": 0.30,
        "cluster_selection_method": "eom",
        "politica_fusion": "raiz_coseno",
        "ari_semillas": (7, 2024, 13, 101),
        "n_finalistas_estabilidad": 8,
        "aspectos_u4": {**_PRISMA["aspectos_u4"], "encuesta_etiquetas": "acreditacion", "hoja_etiquetas": "acreditacion_etiquetas",
                        "archivo_etiquetas": "data/etiquetas_manuales/--Prisma_Validacion_Humana-acreditacion-auditado-14112026.xlsx",   # vigente (decisión Fase 1)
                        "col_respuesta": "respuesta_completa"},
    },
}

if CLAVE_FOCAL not in PERFILES:
    raise KeyError(f"CLAVE_FOCAL={CLAVE_FOCAL!r} no está en PERFILES: {sorted(PERFILES)}")
PERFIL = PERFILES[CLAVE_FOCAL]


def modo_de(clave):
    """'u4' o 'bertopic' efectivo de una pregunta (u4 requiere aspectos_u4['activo'])."""
    p = PERFILES[clave]
    return "u4" if p["modo_aspectos"] == "u4" and p["aspectos_u4"].get("activo", True) else "bertopic"


def params_u4(clave):
    """Parámetros de entrenamiento/carga de U4 de una pregunta (PERFILES[clave]['aspectos_u4'] + semilla, umbral y switches).

    Las rutas de etiquetas (relativas a RAIZ en PERFILES) se devuelven absolutas: así no dependen de la carpeta que
    se pase como `raiz` a entrenar_u4 (que es RAIZ_MODELOS).
    """
    p = PERFILES[clave]
    u4 = dict(p["aspectos_u4"])
    if "RAIZ" in globals():   # RAIZ se define en el Paso 0 (celda de entorno); antes de eso quedan relativas
        import pathlib
        for _k in ("archivo_etiquetas", "carpeta_etiquetas"):
            if u4.get(_k) and not pathlib.Path(u4[_k]).is_absolute():
                u4[_k] = str(RAIZ / u4[_k])
    return {**u4, "semilla": p["semilla_u4"], "umbral_comentario_general": p["umbral_comentario_general"],
            "max_palabras_respuesta_corta": p["max_palabras_respuesta_corta"],
            "reutilizar_modelo": REUTILIZAR_MODELO, "permitir_cpu": PERMITIR_CPU}


def campos_config(clave):
    """Campos de ft.Config que salen del perfil (modo de aspectos, reglas de respuesta corta, residual)."""
    p, r = PERFILES[clave], PERFILES[clave]["residual_hdbscan"]
    return dict(modo_aspectos=modo_de(clave), semilla_u4=p["semilla_u4"],
                max_palabras_respuesta_corta=p["max_palabras_respuesta_corta"],
                umbral_comentario_general=p["umbral_comentario_general"],
                umbral_residual_u4=r["umbral"], min_palabras_residual=r["min_palabras"],
                residual_hdbscan_activo=bool(r["activo"]), residual_rejilla=r["rejilla"],
                residual_minimo=r["minimo_residual"], residual_criterios=dict(r["criterios"]))


# Variables que usa el resto del cuaderno (se derivan del perfil; no se editan a mano).
MIS_ENCUESTAS = [CLAVE_FOCAL]
MODO_ASPECTOS = modo_de(CLAVE_FOCAL)
USAR_CALIBRADOR = PERFIL["usar_calibrador"]
REJILLA = PERFIL["rejilla"]
SELECCION = None                       # None = primera fila que cumple también la estabilidad (ARI >= ari_min)
PERMITIR_NO_CUMPLE = PERFIL["permitir_no_cumple"]
AUDITORIA_DILIGENCIADA = PERFIL["auditoria_diligenciada"]
RUTA_MUESTRA_ANTERIOR = PERFIL["ruta_muestra_anterior"]
print("Encuesta focal:", CLAVE_FOCAL, "| modo de aspectos:", MODO_ASPECTOS)
print("Perfil:", {k: v for k, v in PERFIL.items() if k not in ("aspectos_u4", "residual_hdbscan")})
if MODO_ASPECTOS == "u4":
    print("U4:", params_u4(CLAVE_FOCAL))
    print("Residual:", PERFIL["residual_hdbscan"])


## Paso 0 — Entorno

Instala dependencias, monta Google Drive (solo Colab, primera celda), fija `RAIZ` y el bloque único `RUTAS` (tercera celda; en Colab ajuste `RAIZ_COLAB` en la celda CONFIG) y carga el módulo `nlp_utilidades.modulos_nlp` verificando que la copia en memoria sea la del disco. En local, instale una vez el modelo de spaCy: `python -m spacy download es_core_news_lg` (en Colab se instala solo).


In [ ]:
import sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    !pip install -q bertopic pysentimiento transformers sentence-transformers scikit-learn openpyxl accelerate pyarrow
    !python -m spacy download es_core_news_lg

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:  # termina una celda SIN error (para omitir pasos que no aplican al modo elegido)
    from IPython.core.error import StopExecution
except ImportError:  # fuera de IPython (pruebas)
    class StopExecution(Exception):
        pass

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value.*")

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 180)

AZUL, NARANJA, VERDE, MORADO, ROJO, GRIS = (
    "#2c6fbb", "#c8811d", "#3d8b64", "#7a5195", "#c1453c", "#b8bcc2")
COLOR_SENT = {"POS": VERDE, "NEU": "#9aa0a6", "NEG": ROJO}

GUARDAR_EN_DRIVE = True  # en Colab: salidas y caché quedan en el Drive del proyecto (RAIZ_COLAB); False = /content (temporal)


def raiz_proyecto(marcador="pyproject.toml"):
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / marcador).exists():
            return c
    raise FileNotFoundError(marcador)


# ══ RUTAS: único bloque; todo se deriva de RAIZ (misma estructura del repo en local y en Drive) ══
# El montaje de Drive (solo Colab) está en la primera celda de este paso.
RAIZ = Path(RAIZ_COLAB) if EN_COLAB else raiz_proyecto()
BASE_SALIDA = Path("/content") if (EN_COLAB and not GUARDAR_EN_DRIVE) else RAIZ
SRC = RAIZ / "src"
RUTAS = dict(
    raiz=RAIZ,
    src=SRC,
    modulo=SRC / "nlp_utilidades" / "modulos_nlp.py",
    utilities=RAIZ / "data" / "utilities",                       # léxicos, catálogo y taxonomía
    catalogo=RAIZ / "data" / "utilities" / "preguntas_abiertas.xlsx",
    etiquetas=RAIZ / "data" / "etiquetas_manuales",              # archivos de etiquetas del cliente (fuera de git)
    etapa1=RAIZ / "data" / "processed" / "etapa1",               # corridas de la Etapa 1 (parámetro ruta_etapa1)
    modelos=RAIZ / "models",                                     # contiene modelos/u4_aspectos/<clave>/<run_id>
    cache=BASE_SALIDA / "models" / "cache" / "embeddings_ft",
    calibrador=SRC / "nlp_utilidades" / "calibrador_sentimiento.json",
    salida=BASE_SALIDA / "data" / "processed" / "nlp" / "modelado",   # salidas internas de modelado: <encuesta>/<run_id>/
    nlp=RAIZ / "data" / "processed" / "nlp",                     # 4 tablas del contrato v0.2 (una carpeta <id_corrida_nlp> por corrida)
)
LEXICOS_OBLIGATORIOS = ("stopwords.xlsx", "negaciones.xlsx", "neutralizacion.xlsx", "taxonomia.xlsx")

# Entradas que deben existir (el calibrador es opcional: lo crea el Paso 16).
for _nombre in ("raiz", "src", "modulo", "utilities", "catalogo", "etapa1"):
    assert RUTAS[_nombre].exists(), f"RUTAS['{_nombre}'] no existe: {RUTAS[_nombre]}"
for _archivo in LEXICOS_OBLIGATORIOS:
    assert (RUTAS["utilities"] / _archivo).exists(), f"Falta {_archivo} en {RUTAS['utilities']}"
if MODO_ASPECTOS == "u4":
    assert RUTAS["etiquetas"].exists(), f"No existe la carpeta de etiquetas: {RUTAS['etiquetas']}"
    _etiquetas = RAIZ / PERFIL["aspectos_u4"]["archivo_etiquetas"]
    assert _etiquetas.exists(), f"No existe el archivo de etiquetas de {CLAVE_FOCAL}: {_etiquetas}"
for _nombre in ("modelos", "cache", "salida", "nlp"):  # carpetas de escritura
    RUTAS[_nombre].mkdir(parents=True, exist_ok=True)

# Alias que usa el resto del cuaderno.
DIR_LEXICOS = RUTAS["utilities"]
RUTA_CATALOGO = RUTAS["catalogo"]
RUTA_ETAPA1 = RUTAS["etapa1"]
RAIZ_MODELOS = RUTAS["modelos"]
RUTA_CALIBRADOR = RUTAS["calibrador"]
SALIDA = RUTAS["salida"]
CACHE = RUTAS["cache"]

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Corrida más reciente de la Etapa 1 (sin ruta quemada). Si no hay ninguna, falla con un mensaje claro: corra
# `python run_etapa1.py` desde la raíz del repositorio.
from preparacion.corridas import ultima_corrida

ID_CORRIDA_PREPARACION, CARPETA_CORRIDA_ETAPA1 = ultima_corrida(RUTAS["etapa1"])

print("Proyecto:", RAIZ)
for _nombre, _ruta in RUTAS.items():
    print(f"  {_nombre:<10} {_ruta}")
print("Corrida de la Etapa 1:", ID_CORRIDA_PREPARACION)


In [ ]:
import sys
from pathlib import Path

# El módulo vive en el paquete nlp_utilidades (RUTAS['src'] ya está en sys.path). Importa la clase Config y las
# funciones de entrenamiento y etiquetado de U4.
assert RUTAS["modulo"].exists(), f"No existe el módulo: {RUTAS['modulo']}"
print("Módulo encontrado en:", RUTAS["modulo"])
from nlp_utilidades import modulos_nlp as ft

print("Versión del módulo:", ft.VERSION)
assert ft.VERSION.startswith("ft-r4"), "Suba src/nlp_utilidades/modulos_nlp.py con los cambios de equipo (ft-r4)"
if MODO_ASPECTOS == "u4":
    assert hasattr(ft, "entrenar_u4"), "El módulo cargado no trae los aspectos U4 (ft-r4.15): suba el modulos_nlp.py nuevo y reinicie la sesión"

import hashlib
_huella_disco = hashlib.md5(Path(ft.__file__).read_bytes()).hexdigest()
_huella_memoria = getattr(ft, "HUELLA_ARCHIVO", "(módulo anterior a esta verificación)")
print("Huella en memoria:", _huella_memoria)
print("Huella en disco:  ", _huella_disco)
assert _huella_memoria == _huella_disco, (
    "El módulo cargado NO es el que está en Drive. Entorno de ejecución -> Reiniciar sesión, "
    "y vuelva a correr desde el Paso 0.")

#### 📖 Cómo leer esta salida

**Qué muestra:** que el módulo se encontró, su versión y que la copia en memoria es la misma que está en Drive.
**Normal:** `ft-r4.15` y las dos huellas iguales. **Alarma:** el `assert` falla → *Reiniciar sesión* (no basta con volver a correr la celda: Python no recarga un módulo ya importado).

## Paso 0b — Datos, encoder y calibrador

- **Datos:** el catálogo `preguntas_abiertas.xlsx` vive en `RAIZ/data/utilities`; las respuestas salen de `RAIZ/data/processed/etapa1` (corrida más reciente de la Etapa 1, `RUTAS['etapa1']`); las rutas están en `RUTAS`.
- `ENCODER`: `"potente"` (mpnet, recomendado) o `"rapido"` (MiniLM). Cambiarlo invalida la caché de
  embeddings.
- `USAR_CALIBRADOR` (viene de `PERFILES`, celda CONFIG): si es `True` y existe `src/nlp_utilidades/calibrador_sentimiento.json` (Paso 16), el sentimiento
  sale calibrado; si es `False`, se usa RoBERTuito + margen NEU. Se decide por encuesta con
  validación humana (H28).


In [ ]:
ENCODER = "potente"
# Las rutas (RUTA_CATALOGO, RUTA_ETAPA1, RUTA_CALIBRADOR, DIR_LEXICOS...) vienen del bloque RUTAS (Paso 0, celda de entorno).
# USAR_CALIBRADOR viene de PERFILES (celda CONFIG, Paso 0a).

print("Catálogo:   ", RUTA_CATALOGO)
print("Etapa 1:    ", RUTA_ETAPA1, "| corrida:", ID_CORRIDA_PREPARACION)
print("Léxicos:    ", DIR_LEXICOS)
print("Encoder:    ", ft.ENCODERS[ENCODER])
print("Calibrador: ", (RUTA_CALIBRADOR if RUTA_CALIBRADOR.exists() else "(aún no existe: sin calibrar)")
      if USAR_CALIBRADOR else "DESACTIVADO (USAR_CALIBRADOR = False): RoBERTuito + margen NEU")

#### 📖 Cómo leer esta salida

**Qué muestra:** las cuatro rutas que usará todo el cuaderno. Revise que el catálogo, la carpeta de la Etapa 1 (con su corrida más reciente) y el calibrador existan en **su** Drive.
**Encoder** = el modelo que convierte cada texto en un vector (Paso 5). **Calibrador** = la corrección del sentimiento aprendida de anotaciones humanas (Paso 9).

## Paso 1 — Léxicos

**Qué hace:** carga `stopwords.xlsx`, `negaciones.xlsx`, `neutralizacion.xlsx` y la lista española de NLTK.

`neutralizacion.xlsx` es **obligatorio**: es la única fuente de lo que se retira del texto de aspecto
(VALORATIVO, PERSONA) o no cuenta como contenido (GENERICO). Si falta, o tiene una CATEGORIA
desconocida, la carga se detiene con un mensaje claro.

| Archivo | Afecta |
|---|---|
| `stopwords.xlsx` | Solo las **etiquetas** de los tópicos (c-TF-IDF) |
| `negaciones.xlsx` | Texto de aspecto **y** rama de sentimiento |
| `neutralizacion.xlsx` | Solo el **texto de aspecto** (lo que ve el encoder) |

Cada encuesta recibe NLTK + globales + **sus** particulares + el léxico de neutralización.
Los léxicos son un único juego en `RAIZ/data/utilities` (`DIR_LEXICOS`). La celda de verificación (tras elegir la encuesta focal, Paso 2) usa `encuesta.clave_stopwords` y debe imprimir `True` en las tres líneas.


In [ ]:
recursos = ft.RecursosLexicos.cargar(DIR_LEXICOS)

print(f"Claves de encuesta en stopwords.xlsx: {sorted(recursos.stopwords_por_encuesta)}")
print(f"Palabras que se retiran SIEMPRE del texto de aspecto (valorativos + persona + modificadores de negaciones.xlsx): {len(recursos.terminos_neutralizados())}")
# Conteo por categoría de neutralizacion.xlsx (antes venían del código).
print(f"  valorativos {len(recursos.valorativos)} | persona {len(recursos.persona)} | "
      f"genéricos {len(recursos.genericos)}")

## Paso 2 — Corpus (Etapa 1)

Las respuestas salen de `respuestas.csv` de la **corrida más reciente de la Etapa 1** (`RUTAS['etapa1']`, sin ruta quemada a la corrida): solo las filas con `es_no_respuesta = False` **y** `tiene_alerta = False` (las respuestas con alerta son de la Etapa 1 y **no pasan al NLP**, aunque el término esté negado), con `texto_original` **sin limpiar** (es lo que se segmenta) y `respuesta_id` como llave que viaja hasta `documentos` y `clausulas`. La corrida leída (`id_corrida_preparacion`) queda registrada para `metadatos_nlp`.

Cada pregunta abierta es un modelo independiente ("aspectos a resaltar" y "oportunidades de
mejora" tienen distribuciones opuestas). `MIS_ENCUESTAS` define las encuestas de esta corrida: la
**primera** es la focal (Pasos 3–12). La clave sigue la convención `"{encuesta} :: {pregunta_codigo}"`
del catálogo de la Etapa 1; la celda imprime las disponibles y se detiene si un nombre no coincide (por ejemplo, si la
pregunta focal no está en la corrida: corra `python run_etapa1.py`).


In [ ]:
encuestas = ft.cargar_encuestas_etapa1(RUTA_ETAPA1)
assert {e.id_corrida_preparacion for e in encuestas.values()} == {ID_CORRIDA_PREPARACION}, "las encuestas no vienen de la misma corrida"
print("Corrida de la Etapa 1:", ID_CORRIDA_PREPARACION)

claves_disponibles = list(encuestas)

print("Encuestas disponibles:")
for i, _clave in enumerate(claves_disponibles, start=1):
    print(f"  {i}. {_clave}")


def _resolver_encuesta(nombre):
    """Clave lógica -> clave completa 'encuesta :: pregunta'. Falla si no es única."""
    hallazgos = [c for c in claves_disponibles if c.split(" :: ")[-1] == nombre.strip()]
    if len(hallazgos) != 1:
        raise ValueError(
            f"MIS_ENCUESTAS: '{nombre}' coincide con {len(hallazgos)} encuestas "
            f"(debe ser exactamente 1). Copie la clave lógica tal como aparece arriba."
        )
    return hallazgos[0]


ASIGNADAS = [_resolver_encuesta(n) for n in MIS_ENCUESTAS] if MIS_ENCUESTAS else claves_disponibles
print("\nAsignadas a esta corrida:")
for _clave in ASIGNADAS:
    print(f"  · {_clave}")

#### 📖 Cómo leer esta salida

**Qué muestra:** una línea por pregunta de la corrida de la Etapa 1 (respuestas con comentario). Al final, **Asignadas** = las de `MIS_ENCUESTAS`.
**Alarma:** que la pregunta focal no aparezca (la corrida no la incluye: corra `python run_etapa1.py` con esa encuesta).


In [ ]:
# La focal es la PRIMERA de MIS_ENCUESTAS (celda anterior). Sin input():
# la elección queda escrita en el cuaderno, así cada corrida documenta sobre qué encuesta se hizo.
FOCAL = ASIGNADAS[0]
encuesta = encuestas[FOCAL]
print(f"Focal: {FOCAL} — {len(encuesta.texto_original):,} filas | "
      f"clave de stopwords: '{encuesta.clave_stopwords}'")

In [ ]:
vect = set(recursos.stopwords_vectorizador(encuesta.clave_stopwords))
print(f"Clave de stopwords de la focal: '{encuesta.clave_stopwords}'")
print("¿'del', 'que', 'una' en el vectorizador?   ", {"del", "que", "una"} <= vect)
print("¿'ella', 'doctora', 'excelente' también?   ", {"ella", "doctora", "excelente"} <= vect)
print("¿sin frases (sólo palabras sueltas)?        ", all(" " not in p for p in vect))


#### 📖 Cómo leer esta salida

**Las tres líneas `True`:** el vectorizador de etiquetas (1) quita palabras vacías como *del, que*; (2) quita también valorativos y de persona (*excelente, doctora*), para que no aparezcan en el nombre de un tópico; (3) solo tiene palabras sueltas (las frases de `stopwords.xlsx` se ignoran). Un `False` = léxico mal cargado.
**Conteos:** *palabras que se retiran siempre* = valorativos + persona + modificadores (*no, muy, pero…* de `negaciones.xlsx`). Los **genéricos** van aparte: no se borran en medio del texto, solo se recortan en los bordes y no cuentan como contenido.
**Perilla:** qué palabra va en qué categoría → `neutralizacion.xlsx` (cambia los tópicos) o `stopwords.xlsx` (cambia solo las etiquetas).

## Paso 2.1 — Auditoría de la ingesta (Etapa 1)

Contrasta, para cada pregunta, las respuestas procesables leídas de `respuestas.csv` (con comentario y **sin alerta**) contra `metadatos_preparacion.csv` de la **misma** corrida —`respuestas_con_comentario − respuestas_con_alerta`— (solo conteos; no se imprimen textos) y comprueba que `respuesta_id` no se repite. La tabla trae además, por pregunta, cuántas filas hay (`total`), cuántas son no-respuestas y cuántas tienen alerta (excluidas).

Esta sección no cambia nada del modelo: deja evidencia de qué corrida y cuántas respuestas se leyeron. Ya no hay `alias_redaccion` que resolver: la Etapa 1 entrega el texto en `texto_original`.


In [ ]:
# Fase de APROBACIÓN: lo leído debe coincidir con lo que la Etapa 1 declaró en metadatos_preparacion.csv.
auditoria_ingesta = ft.comparar_con_metadatos_etapa1(encuestas, RUTA_ETAPA1)
auditoria_ingesta.insert(0, "id_corrida_preparacion", ID_CORRIDA_PREPARACION)
auditoria_ingesta["respuesta_id_unicos"] = [encuestas[c].marco["respuesta_id"].is_unique for c in encuestas]
display(auditoria_ingesta)

if (auditoria_ingesta["diferencia"] != 0).any() or not auditoria_ingesta["respuesta_id_unicos"].all():
    print("\n⚠️  Hay preguntas cuyo conteo no coincide con metadatos_preparacion.csv o con respuesta_id repetido. Revise la corrida.")
else:
    print("\n✅ Todas las preguntas coinciden con metadatos_preparacion.csv y sus respuesta_id son únicos.")

#### 📖 Cómo leer esta salida

**Qué muestra:** para cada pregunta, el `total` de filas, las `no_respuestas` y las `con_alerta` (excluidas del NLP), las respuestas `leidas` (procesables) frente a las que declara `metadatos_preparacion.csv` (`respuestas_con_comentario − respuestas_con_alerta`).
**Normal:** `diferencia = 0` y `respuesta_id_unicos = True` en todas. **Alarma:** una diferencia distinta de cero → la corrida cambió o `respuestas.csv` está incompleto; no siga.


In [ ]:
# Fase de CODIFICACIÓN: se deja la auditoría de la ingesta como archivo
# de evidencia junto a las salidas de modelado de esta sesión (`data/processed/nlp/modelado`), para trazabilidad (mismo espíritu
# que auditoria.xlsx / manifest.json del resto del pipeline).
ruta_auditoria_ingesta = SALIDA / "auditoria_ingesta_etapa1.csv"
auditoria_ingesta.to_csv(ruta_auditoria_ingesta, index=False, encoding="utf-8-sig")
print(f"Auditoría de la ingesta guardada en: {ruta_auditoria_ingesta}")

## Paso 3 — Segmentación en cláusulas y motivo

Segmenta cada respuesta (analizador de dependencias de spaCy) y clasifica cada cláusula en un **motivo**:

| Motivo | Ejemplo | Tópico | Sentimiento |
|---|---|---|---|
| `ASPECTO` | "resuelve las dudas con paciencia" | se modela | sí |
| `VALORACION_GLOBAL` | "excelente profesor" | código **-3** | sí |
| `NO_OPINION` | "sin comentarios", "no tengo clase con él" | código **-2** | **no** |
| `RUIDO` | "NR", "-" | código **-2** | no |

**Por qué `VALORACION_GLOBAL` es una categoría y no un tópico:** esas frases no dicen *qué* se valora,
así que no son un aspecto; pero sí son opinión, y su polaridad cuenta en el informe.


In [ ]:
cfg_base = ft.Config(
    output_dir=SALIDA, dir_cache=CACHE, encoder=ft.ENCODERS[ENCODER],
    ruta_calibrador=RUTA_CALIBRADOR if (USAR_CALIBRADOR and RUTA_CALIBRADOR.exists()) else None,
    metodo_segmentacion="dependencias",
    cluster_selection_method=PERFIL["cluster_selection_method"],
    politica_fusion=PERFIL["politica_fusion"],
    ari_semillas=PERFIL["ari_semillas"],
    n_finalistas_estabilidad=PERFIL["n_finalistas_estabilidad"],
    **campos_config(CLAVE_FOCAL),   # modo_aspectos, semilla_u4, respuesta corta, umbrales y residual (ft-r4.15)
)
neutralizador = ft.NeutralizadorAspecto(cfg_base, recursos)

In [ ]:
documentos, clausulas_base = ft.construir_unidades(encuesta, cfg_base, recursos)

#### 📖 Cómo leer esta salida

**Qué muestra (la última línea):** el embudo del corpus. *filas → estados de documento* y *cláusulas → motivos*. Es la línea para citar cuando alguien pregunte "¿con cuántos datos se trabajó?".
El aviso amarillo de spaCy (*Restart to reload dependencies*) es inofensivo. Esta celda tarda ~4 min (análisis sintáctico de todas las respuestas).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
estados = documentos["estado"].value_counts()
ax[0].barh(estados.index, estados.values, color=[GRIS, AZUL, NARANJA, ROJO][: len(estados)])
ax[0].set_title("Filas del corpus por estado")
for i, v in enumerate(estados.values):
    ax[0].text(v, i, f" {v:,} ({100*v/len(documentos):.1f}%)", va="center", fontsize=8)

motivos = clausulas_base["motivo"].value_counts()
ax[1].barh(motivos.index, motivos.values, color=[VERDE, MORADO, NARANJA, GRIS][: len(motivos)])
ax[1].set_title("Cláusulas por motivo")
for i, v in enumerate(motivos.values):
    ax[1].text(v, i, f" {v:,} ({100*v/len(clausulas_base):.1f}%)", va="center", fontsize=8)
plt.tight_layout(); plt.show()

print(f"Cláusulas por documento con opinión: "
      f"{len(clausulas_base) / max((documentos['estado'] == 'CON_OPINION').sum(), 1):.2f}")

#### 📖 Cómo leer esta salida

**Qué muestra:** el mismo embudo en barras. *Cláusulas por documento* ≈ 1,7 → la respuesta típica contiene 1 o 2 ideas: por eso se modela por cláusula y no por respuesta completa.

In [ ]:
print("Muestra aleatoria de NO_OPINION (debe ser 100% no evaluativa):")
no_op = clausulas_base[clausulas_base["motivo"] == "NO_OPINION"]
for t in no_op["clausula"].sample(min(15, len(no_op)), random_state=1):
    print("   ·", t[:110])

print("\nMuestra aleatoria de VALORACION_GLOBAL (debe ser elogio/crítica sin aspecto):")
glob_ = clausulas_base[clausulas_base["motivo"] == "VALORACION_GLOBAL"]
for t in glob_["clausula"].sample(min(15, len(glob_)), random_state=1):
    print("   ·", t[:110])

#### 📖 Cómo leer esta salida

**Es una revisión humana rápida del filtro**, no un resultado. Lea las dos listas:
- `NO_OPINION` debería ser **100 %** no evaluativa. Si aparece una opinión real, el filtro la está botando (falso positivo). Como referencia, en obs19 apareció 1 de 15: tolerable, pero anótelo si se repite.
- `VALORACION_GLOBAL` debería ser elogio/crítica **sin** aspecto. Si aparecen fragmentos sin sentido (restos de un corte), es el segmentador, no el filtro.
Cada ejecución saca otra muestra aleatoria.

**Qué revisar aquí:** si en la muestra de `NO_OPINION` aparece una opinión real, o en
`VALORACION_GLOBAL` aparece un aspecto ("es muy puntual"), anótelo. Se corrige editando los léxicos
(o `neutralizacion.xlsx`), sin tocar código.


### Casos límite de la segmentación (H10)

La ruta de producción es la de **dependencias** (spaCy). Esta celda compara, sobre los mismos textos,
el segmentador por **conectores** (`Segmentador`: puntuación fuerte + adversativos de `negaciones.xlsx`;
sin spaCy, determinista, rápido) con `SegmentadorDependencias`. No altera el pipeline: sirve para revisar
a ojo dónde difieren (columna `coinciden`) antes de confiar en una versión nueva del módulo o del modelo.


In [ ]:
# Casos límite (H10): compara las dos rutas de segmentación. Es SOLO diagnóstico:
# construir_unidades() usa siempre la ruta de dependencias (cfg_base.metodo_segmentacion).
segmentador_conectores = ft.Segmentador(cfg_base, recursos)       # puntuación + adversativos (sin spaCy)
segmentador_dependencias = ft.crear_segmentador(cfg_base)         # estructura sintáctica (spaCy)

casos = [
    "Con el Dr. Jairo uno no se siente estudiante. Su método es inigualable",
    "Explica muy bien los temas, pero nunca responde los correos",
    "Aunque llega tarde, sus clases son excelentes",
    "Ninguna, muy buena clase",
    "Ningún aspecto por mejorar",
    "La profe Merly explica muy bien, es una excelente profesora",
    "NO TENGO CLASE CON ESTE DOCENTE",
    "el profesor es muy grosero",
    "La universidad tiene que volverse más ágil desde lo administrativo "
    "(incluido en ello lo financiero y lo jurídico).",
    "El profesor explica bien y espera que los estudiantes participen.",
    "1) Realizar visitas a instituciones como DNP, DANE, IGAC, ANT, IGAC, UPRA "
    "2) Participación de funcionario y ex funcionario de las instituciones encargadas de "
    "Planeación en el país durante las clases "
    "3) Que se generen espacios con universidades nacionales, extranjeras para compartir experiencias",
    "1. Etc.",  # documento que empieza y termina en fragmentos cortos
]

filas = []
for texto in casos:
    por_conectores = segmentador_conectores.segmentar(texto)
    por_dependencias = segmentador_dependencias.segmentar(texto)
    filas.append({
        "respuesta": texto[:60],
        "n_conectores": len(por_conectores),
        "n_dependencias": len(por_dependencias),
        "coinciden": por_conectores == por_dependencias,
        "conectores": " | ".join(por_conectores),
        "dependencias": " | ".join(por_dependencias),
    })
pd.DataFrame(filas)


## Paso 3.1 — Taxonomía por encuesta

Cada encuesta trae su propia taxonomía, diseñada por un experto, en `taxonomia.xlsx` (hoja
`aspectos`, columnas `pregunta_codigo`, `aspecto_id`, `aspecto_nombre`, `descripcion`; se busca por la clave de la pregunta). `cfg.taxonomia` alimenta
`aspecto_sugerido` (Paso 8b): es ayuda para **nombrar** tópicos, no cambia el modelado. **Solo se usa en el modo `bertopic`: en modo `u4` este paso se omite.**

Si una encuesta activa no tiene taxonomía, la celda se detiene: el equipo decide si agrega sus filas
al Excel o excluye la encuesta. No hay taxonomía genérica de respaldo.


In [ ]:
# Solo modo bertopic: la taxonomía es ayuda para NOMBRAR tópicos. En modo u4 el aspecto lo decide el clasificador
# (sus clases salen del archivo de etiquetas) y este paso se omite.
if MODO_ASPECTOS == "u4":
    print("Paso omitido: modo_aspectos = 'u4' (la taxonomía solo se usa en el modo 'bertopic').")
    raise StopExecution

# Fase de APROBACIÓN: no se sigue si alguna pregunta ACTIVA del catálogo no tiene taxonomía en la hoja `aspectos`
# de taxonomia.xlsx. A propósito no hay respaldo genérico (ver comentario de TAXONOMIA_DEFECTO en modulos_nlp.py):
# si esto falla, hay que decidir en equipo -- agregar la taxonomía faltante o excluir esa pregunta -- no
# "arreglarlo" en código.
taxonomias = ft.cargar_taxonomias(DIR_LEXICOS)

activas_catalogo = set(ft.clave_logica(clave) for clave in encuestas)
faltan_taxonomia = activas_catalogo - set(taxonomias)

print("Preguntas activas en este corpus:", sorted(activas_catalogo))
print("Preguntas con taxonomía en taxonomia.xlsx:", sorted(taxonomias))

if faltan_taxonomia:
    raise KeyError(
        f"Faltan taxonomías para: {sorted(faltan_taxonomia)}. "
        "Agregue sus filas a la hoja `aspectos` de taxonomia.xlsx (pregunta_codigo/aspecto_id/aspecto_nombre/descripcion) "
        "antes de continuar, o retire esa pregunta de las activas en preguntas_abiertas.xlsx."
    )
print("\n✅ Todas las preguntas activas tienen taxonomía. Se puede continuar.")

In [ ]:
if MODO_ASPECTOS == "u4":
    print("Paso omitido: modo_aspectos = 'u4'.")
    raise StopExecution

import dataclasses

cfg_base = dataclasses.replace(cfg_base, taxonomia=taxonomias[ft.clave_logica(CLAVE_FOCAL)])

print(f"Taxonomía activa para '{CLAVE_FOCAL}': {len(cfg_base.taxonomia)} códigos\n")
for codigo, descripcion in cfg_base.taxonomia.items():
    print(f"  {codigo:<32} {descripcion}")

#### 📖 Cómo leer esta salida

**Qué muestra:** los códigos de la taxonomía de la encuesta con su descripción. **No clasifican nada:** solo se usan en el Paso 8b para *sugerir* un nombre a cada tópico descubierto.

## Paso 4 — Texto de aspecto (H3)

**Qué hace:** la rama de tópicos no ve la cláusula original sino su versión neutralizada.

**Por qué:** el encoder agrupa por similitud global. En respuestas cortas lo dominante es la
valoración ("excelente", "no", "pero") y la persona ("ella", "doctora"), así que agruparía por tono
y por género. Retirando esas palabras (definidas en `neutralizacion.xlsx`), lo que queda es
**de qué se habla**.

La rama de sentimiento sigue leyendo la cláusula original, con tildes y negadores.


In [ ]:
aspecto = clausulas_base[clausulas_base["motivo"] == "ASPECTO"]
aspecto.sample(15, random_state=3)[["clausula", "texto_aspecto", "n_contenido"]]

#### 📖 Cómo leer esta salida

**Qué muestra:** una muestra de cláusulas junto a su **texto de aspecto**, que es lo que el modelo de tópicos realmente "lee": sin valoraciones, sin personas, sin negadores. `n_contenido` = palabras con contenido que quedaron.
**Alarma:** filas con `n_contenido = 1` cuya única palabra no nombra un aspecto (un verbo suelto, un adjetivo). Esas son las que terminan en tópicos cajón de sastre.
**Perilla:** `neutralizacion.xlsx`.

## Paso U — Aspectos supervisados U4 (`modo_aspectos = "u4"`)

**Qué cambia.** El aspecto de cada cláusula con motivo ASPECTO ya **no** lo deciden UMAP/HDBSCAN: lo decide **U4**, un clasificador entrenado *por pregunta* solo con **etiquetas auditadas a mano** (encoder mpnet con las últimas 4 capas descongeladas + cabeza lineal; texto de entrada = la cláusula ORIGINAL, no la neutralizada). El sentimiento (RoBERTuito + calibrador) y el ABSA no cambian. Los Pasos 5–8b solo corren con `modo_aspectos = "bertopic"` y se omiten solos.

| Paso | Qué hace |
|---|---|
| **U1** | Entrena (o reutiliza) el modelo U4 de la pregunta, lo evalúa **una sola vez** en el hold-out y lo guarda en `RAIZ/models/modelos/u4_aspectos/<clave>/<run_id>/` |
| **U2** | Etiqueta **todo** el corpus de la pregunta y aplica las reglas («Comentario General», residual) |
| **U3** | Resultados: matriz de confusión del hold-out, distribución de `p_aspecto_u4`, tabla de aspectos, % «Comentario General» y % `respuesta_corta`, y un ejemplo de unión `clausulas` ⟕ `documentos` por `doc_id` |
| **U4** | Residual: tamaño y diagnóstico del umbral |
| **U5–U6** | Búsqueda y modelo UMAP/HDBSCAN **solo sobre el residual**; tabla de tópicos residuales |
| **U7** | Modelo final (`procesar_encuesta`) y conciliación |

**Precedencia por cláusula:** (a) NO_OPINION y RUIDO conservan su tratamiento; (b) en una **respuesta corta** (≤ `max_palabras_respuesta_corta` palabras de la respuesta completa sin limpiar) toda cláusula con opinión pasa por U4 y, si su probabilidad máxima es menor que `umbral_comentario_general`, queda como **«Comentario General»**; si no, recibe el aspecto de U4; (c) la columna `respuesta_corta` marca todas las cláusulas y documentos de respuestas cortas; (d) las cláusulas **largas** (≥ 6 palabras) con probabilidad menor que el umbral del residual forman el **residual**: pasan por UMAP/HDBSCAN para descubrir aspectos nuevos y su aspecto final es **«Por descubrir»** (se conservan `aspecto_u4` y todas las `p_u4_*`); (e) las VALORACION_GLOBAL de respuestas largas siguen como VALORACION_GLOBAL.
*Caso sin regla:* una cláusula de ≤ 5 palabras dentro de una respuesta larga no es residual (le faltan palabras) ni «Comentario General» (la respuesta no es corta): conserva el aspecto de U4 y, si su probabilidad es baja, se marca `u4_baja_confianza`.

**Archivos que deben existir** (uno por pregunta, mismo formato que el de obs19): `RAIZ/data/etiquetas_manuales/` (el archivo exacto de cada pregunta se fija en `PERFILES[clave]['aspectos_u4']['archivo_etiquetas']`), con las columnas `id`, `revisor`, la de respuesta, `fragmento_final_entrenamiento` y `aspecto_final_entrenamiento`. El vocabulario de aspectos de cada pregunta son las clases de ese archivo (se normalizan mayúsculas, tildes y espacios; si el auditor escribió una errata, aparece como una clase más: revise la lista de clases del Paso U1). **No hay respaldo:** si falta el archivo auditado o alguna columna, el entrenamiento se detiene con un error que dice qué archivo y qué columnas esperaba; nunca se entrena con hojas sin auditar.

> ⚠️ **Todavía no hay ninguna métrica real de U4 en este cuaderno.** Las cifras (acierto, F1, % de «Comentario General», tamaño del residual) aparecen solo cuando usted ejecute los pasos en Colab con GPU. Los umbrales 0,50 son **provisionales**.

## Paso U1 — Entrenar (o reutilizar) el modelo U4

`ft.entrenar_u4` carga el archivo **auditado** de la pregunta (error claro si falta), crea y comprueba las carpetas de salida **antes** de gastar GPU, separa un **hold-out agrupado** (15 % de los grupos respuesta–fragmento; mismos cortes que el experimento de regresión logística), entrena con parada temprana sobre una validación interna agrupada, evalúa **una sola vez** en el hold-out y guarda el modelo (solo los parámetros entrenables + metadatos) verificando que al recargarlo da las mismas predicciones. La semilla de entrenamiento es `semilla_u4 = 42`.
Con `REUTILIZAR_MODELO = True` no reentrena si ya existe un modelo con la misma huella de datos + configuración; con `SOLO_ETIQUETAR = True` solo carga el último modelo guardado.

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

if SOLO_ETIQUETAR:
    modelo_u4 = ft.modelo_u4_de_clave(RAIZ_MODELOS, CLAVE_FOCAL)
    print("SOLO_ETIQUETAR: modelo cargado SIN reentrenar.")
else:
    modelo_u4 = ft.entrenar_u4(CLAVE_FOCAL, params_u4(CLAVE_FOCAL), RAIZ_MODELOS,
                               reutilizar=REUTILIZAR_MODELO, permitir_cpu=PERMITIR_CPU)
print(f"\nModelo U4: {modelo_u4.run_id} | reutilizado: {modelo_u4.reutilizado}")
print("Carpeta:   ", modelo_u4.carpeta)
print("Clases (orden = código de aspecto):", {i: c for i, c in enumerate(modelo_u4.clases)})
display(pd.Series({k: v for k, v in modelo_u4.metricas.items() if k != "por_clase"}).to_frame("valor"))
display(pd.DataFrame(modelo_u4.metricas.get("por_clase", [])))

#### 📖 Cómo leer esta salida

**Qué muestra:** el run del modelo, su carpeta y el **orden de clases**, que es el código de aspecto (`topico`) de todo el pipeline: no cambia entre corridas mientras no cambien los datos auditados. Luego, las métricas del **hold-out** (`acierto_holdout`, `f1_macro_holdout`, `n_holdout_evaluable`) y, por clase, precisión / recall / F1 con su `n`.
**Normal:** una clase con `n` < 10 aparece como `NO CONCLUYENTE` (no se puede juzgar con tan pocos casos). **Alarma:** `NO HAY GPU` (cambie el tipo de entorno), `ErrorEtiquetasU4` (falta el archivo auditado, una columna o hay una etiqueta fuera del vocabulario: el mensaje lo dice) o una clase con recall 0 y `n` grande.
Las métricas son de **una sola evaluación** en el hold-out; no se usaron para elegir nada.

In [ ]:
# Desempeño en conjunto de entrenamiento

# Por clase
_p = ft.ParametrosU4.desde(params_u4(CLAVE_FOCAL), CLAVE_FOCAL)
_d = ft.cargar_datos_u4(RAIZ, _p, CLAVE_FOCAL)
_tr, _va, _ = ft.particionar_datos_u4(_d, _p.hip())
display(ft._metricas_holdout(_d.tabla["y"].to_numpy()[_tr], modelo_u4.predecir_proba(_d.tabla["texto"].iloc[_tr].tolist()), _d.clases, 10)["por_clase"])

In [ ]:
_tr = ft.particionar_datos_u4(_d, _p.hip())[0]
_m = ft._metricas_holdout(_d.tabla["y"].to_numpy()[_tr], modelo_u4.predecir_proba(_d.tabla["texto"].iloc[_tr].tolist()), _d.clases, 10)
display(pd.Series({"acierto_entrenamiento": _m["accuracy"], "f1_macro_entrenamiento": _m["f1_macro"], "f1_macro_entrenamiento_todas_las_clases": _m["f1_macro_todas_las_clases"], "n_entrenamiento_evaluado": _m["n_holdout_evaluable"]}).to_frame("valor"))


## Paso U2 — Etiquetar todo el corpus de la pregunta

`ft.etiquetar_corpus_u4` etiqueta **todas** las cláusulas ASPECTO (y las VALORACION_GLOBAL de respuestas cortas) con el texto original de la cláusula, en lotes de 128 y deduplicando textos idénticos, guarda la probabilidad de **todas** las clases (`p_u4_<aspecto>`) y aplica las reglas del Paso U. También marca `u4_visto_en_entrenamiento` (coincidencia **aproximada** de texto con los fragmentos usados para entrenar/validar, sin contar el hold-out) para poder excluir esas filas al evaluar.
Puede correrse **sin reentrenar** (`SOLO_ETIQUETAR = True`). Escribe además los diagnósticos de umbral en el Excel de entrenamiento (hojas `diag_cg_<fecha-hora>` y `diag_res_<fecha-hora>`: cada etiquetado añade las suyas y conserva las anteriores).

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

import dataclasses

cfg_u4 = dataclasses.replace(cfg_base, nombre=FOCAL, ruta_modelo_u4=Path(modelo_u4.carpeta))
etapas = ft.etiquetar_corpus_u4(modelo_u4, encuesta, documentos, clausulas_base, cfg_u4)

cl_u4 = etapas.clausulas
resumen_asig = (cl_u4["asignacion"].fillna("(sin asignar)").value_counts().rename("clausulas").to_frame())
resumen_asig["%"] = (100 * resumen_asig["clausulas"] / len(cl_u4)).round(2)
display(resumen_asig)
print("\nDIAGNÓSTICO del umbral de «Comentario General» (umbral actual:", cfg_u4.umbral_comentario_general, "— PROVISIONAL):")
display(etapas.diagnostico_cg["umbrales"])

#### 📖 Cómo leer esta salida

**Primera tabla:** a qué mecanismo debe cada cláusula su aspecto: `u4` (aspecto de U4), `regla_comentario_general` (respuesta corta con probabilidad baja), `residual_hdbscan` («Por descubrir»), `valoracion_global`, `no_opinion`, `ruido`.
**Segunda tabla (diagnóstico):** para cada umbral (0,3 a 0,7), cuántas cláusulas de respuestas cortas pasarían a «Comentario General». Sirve para **ajustar** `umbral_comentario_general` en `PERFILES`: un umbral alto manda mucho a «Comentario General»; uno bajo deja casi todo con el aspecto de U4. Contexto: en obs19 los humanos asignaron un aspecto concreto en la gran mayoría de las respuestas de ≤5 palabras (p. ej. «Conoce del tema» → Dominio disciplinar), por eso la regla usa la probabilidad de U4 y no reetiqueta todo.
**Alarma:** que casi todo `u4` caiga en `regla_comentario_general` (umbral demasiado alto o modelo poco seguro).

In [ ]:
# Diagnóstico de distintos valores del umbral `residual_hdbscan`

_cl = etapas.clausulas
_asp = _cl[_cl["motivo"] == "ASPECTO"]
_eleg = _asp[(_asp["clausula"].astype(str).str.split().str.len() >= cfg_u4.min_palabras_residual) & ~_asp["respuesta_corta"].astype(bool)]
_umbrales = sorted({0.3, 0.4, 0.5, 0.6, 0.7, round(float(cfg_u4.umbral_residual_u4), 4)})
_diag_res = pd.DataFrame({
    "umbral": _umbrales,
    "n_clausulas_elegibles": len(_eleg),
    "n_pasarian_a_residual": [int((_eleg["p_aspecto_u4"] < u).sum()) for u in _umbrales],
    "pct_residual": [round(100 * float((_eleg["p_aspecto_u4"] < u).mean()), 2) for u in _umbrales],
    "es_el_umbral_actual": [abs(u - cfg_u4.umbral_residual_u4) < 1e-9 for u in _umbrales],
})
print(f"DIAGNÓSTICO del umbral del residual (umbral actual: {cfg_u4.umbral_residual_u4} — PROVISIONAL):")
display(_diag_res)


#### 📖 Cómo leer esta salida

Cómo leerla:

`n_clausulas_elegibles`: cláusulas ASPECTO con ≥ 6 palabras y que no son de una respuesta corta. Es el denominador, igual en todas las filas.
`pct_residual`: el porcentaje de esas elegibles que pasaría a «Por descubrir» con cada umbral. Es un porcentaje de las elegibles, no del corpus completo.

Esta tabla es una versión simplificada de `etapas.diagnostico_residual`, que ya calcula el Paso U4 y añade dos cosas: el porcentaje sobre todas las cláusulas ASPECTO (pct_del_corpus_aspecto) y el acierto de U4 por debajo y por encima de cada umbral con el hold-out. Esa segunda parte es la más útil para elegir el umbral, porque te dice si las probabilidades bajas realmente corresponden a errores. Si quieres ver todo junto, usa display(etapas.diagnostico_residual).

Para el residual, el umbral 0.50 deja de ser el que decide solo: también cuenta si el tamaño resultante supera el mínimo minimo_residual (por defecto 300) para que se ejecute HDBSCAN.

## Paso U3 — Resultados de U4

Matriz de confusión del hold-out (conteos y normalizada por fila), distribución de `p_aspecto_u4`, tabla de aspectos, porcentaje de «Comentario General» y de `respuesta_corta`, y un ejemplo de unión `clausulas` ⟕ `documentos` por `doc_id` (la llave única de `documentos`).

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

_xlsx = Path(modelo_u4.carpeta) / "entrenamiento_u4.xlsx"
try:
    cc = pd.read_excel(_xlsx, sheet_name="confusion_conteos", index_col=0)
    cn = pd.read_excel(_xlsx, sheet_name="confusion_normalizada", index_col=0)
except Exception as exc:  # noqa: BLE001
    cc = cn = None
    print(f"No pude leer las matrices de {_xlsx.name} ({type(exc).__name__}: {exc})")

if cc is not None:
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    for a, m, titulo, fmt in ((ax[0], cc, "Conteos (hold-out)", "d"), (ax[1], cn, "Normalizada por fila = recall", ".2f")):
        im = a.imshow(m.values, cmap="Blues", vmin=0)
        a.set_xticks(range(len(m.columns))); a.set_xticklabels(m.columns, rotation=60, ha="right", fontsize=7)
        a.set_yticks(range(len(m.index))); a.set_yticklabels(m.index, fontsize=7)
        a.set_xlabel("predicho por U4"); a.set_ylabel("etiqueta auditada"); a.set_title(titulo)
        for i_ in range(m.shape[0]):
            for j_ in range(m.shape[1]):
                v = m.values[i_, j_]
                if v:
                    a.text(j_, i_, format(v, fmt), ha="center", va="center", fontsize=6,
                           color="white" if v > m.values.max() * 0.6 else "black")
        plt.colorbar(im, ax=a, fraction=0.04)
    plt.tight_layout(); plt.show()

pas = cl_u4[cl_u4["asignacion"].isin(["u4", "regla_comentario_general", "residual_hdbscan"])]
corta = pas[pas["respuesta_corta"]]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].hist(pas["p_aspecto_u4"], bins=40, color=AZUL)
ax[0].axvline(cfg_u4.umbral_residual_u4, ls="--", c=ROJO, lw=1, label="umbral del residual")
ax[0].set_title("p_aspecto_u4 · todas las cláusulas con U4"); ax[0].legend(fontsize=8)
ax[1].hist(corta["p_aspecto_u4"], bins=40, color=NARANJA)
ax[1].axvline(cfg_u4.umbral_comentario_general, ls="--", c=ROJO, lw=1, label="umbral «Comentario General»")
ax[1].set_title("p_aspecto_u4 · cláusulas de respuestas cortas"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

n_suj = max(len(pas), 1)
print(f"Cláusulas sujetas a aspecto: {len(pas):,} | «Comentario General»: {int((pas['asignacion'] == 'regla_comentario_general').sum()):,} "
      f"({100 * (pas['asignacion'] == 'regla_comentario_general').sum() / n_suj:.1f} %)")
print(f"respuesta_corta: {100 * cl_u4['respuesta_corta'].mean():.1f} % de las cláusulas · "
      f"{100 * etapas.documentos['respuesta_corta'].mean():.1f} % de las filas del corpus (≤ {cfg_u4.max_palabras_respuesta_corta} palabras)")
display(ft.tabla_aspectos_u4(modelo_u4, cl_u4)[["topico", "aspecto", "clausulas", "documentos", "p_media", "p_mediana", "especial", "ejemplos"]])

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

# Unión clausulas ⟕ documentos por doc_id (llave única en documentos): trae la respuesta completa a cada cláusula.
ejemplo = cl_u4.merge(etapas.documentos[["doc_id", "respuesta_completa", "n_palabras_respuesta"]],
                      on="doc_id", how="left", validate="many_to_one")
assert len(ejemplo) == len(cl_u4) and ejemplo["respuesta_completa"].notna().all(), "la unión por doc_id perdió filas"
display(ejemplo[["doc_id", "clausula", "aspecto", "aspecto_u4", "p_aspecto_u4", "respuesta_corta", "n_palabras_respuesta", "respuesta_completa"]]
        .sample(10, random_state=1))

#### 📖 Cómo leer esta salida

**Matrices de confusión (hold-out):** filas = etiqueta auditada, columnas = lo que predijo U4. La diagonal son los aciertos; la matriz **normalizada por fila** es el recall de cada clase. Fíjese en las celdas fuera de la diagonal grandes: son los pares de aspectos que U4 confunde.
**Histogramas:** distribución de la probabilidad máxima. Las cláusulas a la izquierda de la línea roja son las de baja confianza (a «Comentario General» si la respuesta es corta, al residual si la cláusula es larga).
**Tabla de aspectos:** una fila por clase de U4 (más «Comentario General» y «Por descubrir» si tienen cláusulas). `p_media` / `p_mediana` = seguridad típica del modelo en ese aspecto. `especial = True`: no es un aspecto concreto (p. ej. «Opinión general»): se reporta, no se prioriza en el ABSA.
**Ejemplo de unión:** `documentos.csv` y `clausulas.csv` se unen por `doc_id`; `respuesta_completa` es el texto original sin limpiar. La cláusula no siempre es una subcadena literal de la respuesta (las reglas de reparación de fragmentos pueden reconstruirla).
**Importante:** filtre `u4_visto_en_entrenamiento == False` si va a medir U4 contra etiquetas humanas sobre el corpus.

## Paso U4 — Residual: tamaño y diagnóstico del umbral

El **residual** son las cláusulas con motivo ASPECTO, **más de 5 palabras** (`min_palabras`), probabilidad máxima de U4 **menor que el umbral** (`residual_hdbscan['umbral']`, 0,50 provisional) y que **no** son de una respuesta corta. Sobre ellas se busca descubrir aspectos nuevos; su aspecto final es **«Por descubrir»** (distinto de la clase «Opinión general»/«Sin aspecto» de U4).
La tabla de diagnóstico dice, para cada umbral, qué % del corpus sería residual y —con las predicciones del hold-out— **cuánto acierta U4 por debajo y por encima del umbral**: si por debajo acierta poco, las probabilidades bajas sí corresponden a errores y el residual es un buen filtro.

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

mask_res = cl_u4["es_residual"]
minimo_res = cfg_u4.residual_minimo if cfg_u4.residual_minimo is not None else ft.MINIMO_MODELABLES
print(f"Residual: {int(mask_res.sum()):,} cláusulas ({100 * mask_res.sum() / max((cl_u4['motivo'] == 'ASPECTO').sum(), 1):.1f} % de las cláusulas ASPECTO) | "
      f"umbral {cfg_u4.umbral_residual_u4} · ≥ {cfg_u4.min_palabras_residual} palabras | mínimo para ejecutar HDBSCAN: {minimo_res}")
if not cfg_u4.residual_hdbscan_activo:
    print("residual_hdbscan_activo = False: no hay residual (toda cláusula ASPECTO conserva el aspecto de U4).")
dr = etapas.diagnostico_residual
display(dr)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(dr["umbral"], dr["pct_del_corpus_aspecto"], "o-", color=AZUL)
ax[0].set_xlabel("umbral de probabilidad"); ax[0].set_ylabel("% de cláusulas ASPECTO que serían residual")
ax[0].axvline(cfg_u4.umbral_residual_u4, ls="--", c=ROJO, lw=1); ax[0].set_title("Tamaño del residual según el umbral")
if "holdout_acierto_bajo" in dr:
    ax[1].plot(dr["umbral"], dr["holdout_acierto_bajo"], "o-", color=ROJO, label="acierto de U4 con p < umbral")
    ax[1].plot(dr["umbral"], dr["holdout_acierto_sobre"], "o-", color=VERDE, label="acierto de U4 con p ≥ umbral")
    ax[1].set_ylim(0, 1); ax[1].set_xlabel("umbral de probabilidad"); ax[1].legend(fontsize=8)
    ax[1].set_title("¿Las probabilidades bajas son errores? (hold-out)")
plt.tight_layout(); plt.show()

#### 📖 Cómo leer esta salida

**Izquierda:** a mayor umbral, más cláusulas caen al residual. **Derecha (hold-out):** la curva roja debe quedar **claramente por debajo** de la verde; si ambas son parecidas, la probabilidad de U4 no separa aciertos de errores y el residual mezclaría cláusulas bien clasificadas (suba la exigencia o apague el residual con `activo=False`). `holdout_n_bajo` indica cuántos casos sostienen cada cifra: con pocos, no concluya.
**Perilla:** `residual_hdbscan['umbral']` y `['min_palabras']` en `PERFILES`. Con menos cláusulas que `minimo_residual` no se ejecuta HDBSCAN: todas quedan «Por descubrir» con `topico_residual = -1` (en las cláusulas que no son residual, `topico_residual` va vacío).

## Paso U5 — Búsqueda y modelo del residual (UMAP → HDBSCAN)

Se reutiliza la maquinaria de siempre (`GridSearchAspectos` / `ModeladorAspectos`) **solo sobre el residual**, con los embeddings mpnet **originales** del texto de aspecto neutralizado (no los de U4), rejilla `residual_hdbscan['rejilla']`, `permitir_no_cumple=True` y **criterios de negocio relajados** (`residual_hdbscan['criterios']`: pocos tópicos, silueta ≥ 0, ARI ≥ 0,5): el residual es pequeño y heterogéneo por construcción, y exigirle lo mismo que al corpus completo lo haría fallar siempre. Las puertas de silueta/ARI/outliers del scorecard general quedan `NO_APLICA`; estas métricas se reportan aparte.

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

if "codificador" not in globals():
    codificador = ft.EncoderSentenceTransformers(cfg_u4)
etapas = ft.modelar_residual_u4(etapas, cfg_u4, recursos, codificador=codificador,
                                clave_stopwords=encuesta.clave_stopwords)
display(pd.Series({k: v for k, v in etapas.metricas_residual.items() if k != "criterios"}).to_frame("valor"))

#### 📖 Cómo leer esta salida

**Métricas del residual:** `n_clausulas`, `n_topicos`, `pct_outliers` (cláusulas sin grupo tras fusión y reasignación), `silueta` (separación de los grupos) y `ari_medio` (estabilidad entre semillas). `ejecutado = False` + `motivo_no_ejecutado` explica por qué no se modeló (apagado, vacío o por debajo del mínimo).
**Normal:** pocos grupos y un % de outliers moderado. **Alarma:** muchos outliers o silueta cercana a 0: los grupos no son confiables como aspectos nuevos; léalos como pistas, no como resultados.

## Paso U6 — Tópicos residuales: ¿aspecto nuevo o aspecto existente mal reconocido?

Cada tópico residual se describe con sus **términos c-TF-IDF**, su **aspecto U4 dominante** (el `aspecto_u4` más frecuente entre sus cláusulas) y su **pureza** (% de cláusulas con ese aspecto). Un tópico con aspecto dominante muy puro suele ser un aspecto **existente mal reconocido**; uno con pureza baja y términos propios sugiere un aspecto **nuevo**.

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

tr_res = etapas.tabla_residual
if tr_res.empty:
    print("No hay tópicos residuales (residual vacío o apagado).")
else:
    display(tr_res[["topico_residual", "etiqueta", "clausulas", "documentos", "cohesion",
                    "aspecto_u4_dominante", "pureza_u4", "terminos", "ejemplos"]])
    graf = tr_res.sort_values("clausulas")
    fig, ax = plt.subplots(figsize=(11, max(3, 0.4 * len(graf))))
    ax.barh(graf["etiqueta"].str[:60], graf["clausulas"], color=[GRIS if t < 0 else MORADO for t in graf["topico_residual"]])
    ax.set_title("Cláusulas por tópico residual (gris = sin grupo)")
    plt.tight_layout(); plt.show()

#### 📖 Cómo leer esta salida

**Tabla:** una fila por tópico residual (la fila `-1` = cláusulas sin grupo, con el resumen del modelado). Lea los `ejemplos` antes de fiarse de los `terminos`. `pureza_u4` alta (≥ 70 %) → probablemente un aspecto de la lista que U4 reconoció con poca seguridad; baja → candidato a aspecto nuevo para el equipo.
**Gráfica:** tamaño de cada grupo. La decisión de convertir un grupo en aspecto nuevo es **humana**: hay que auditarlo y, si se acepta, añadirlo a los datos auditados y reentrenar.

## Paso U7 — Modelo final y conciliación (modo u4)

`procesar_encuesta` reutiliza las etapas U2 y U5 (no recalcula nada), calcula el sentimiento de cada cláusula con opinión (RoBERTuito + calibrador, **sin cambios**) y construye la tabla de aspectos, la matriz ABSA y el scorecard. **Todas las filas `chequeo` de la conciliación deben valer 1** (incluidas las cláusulas «Comentario General» y «Por descubrir»).

In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

cfg_final = cfg_u4
clasificador = ft.ClasificadorRobertuito(cfg_final)

resultado = ft.procesar_encuesta(
    encuesta, cfg_final, recursos,
    codificador=codificador, clasificador=clasificador,
    unidades=(documentos, clausulas_base), modelo_u4=modelo_u4, etapas_u4=etapas,
)
print("Modo de aspectos:", resultado.modo, "| modelo U4:", resultado.u4["modelo_u4_id"],
      "| sentimiento calibrado:", resultado.metricas["sentimiento_calibrado"])
resultado.conciliacion

#### 📖 Cómo leer esta salida

**Conciliación:** cada `chequeo` debe valer **1**. En modo u4 las cuentas son: ASPECTO + VALORACION_GLOBAL de respuestas cortas = aspectos asignados + «Comentario General» (-5) + «Por descubrir» (-6); VALORACION_GLOBAL = código -3 + las globales de respuestas cortas (que pasan por U4). Códigos: `0…K-1` = aspectos de U4 (orden de `clases.json`), `-2` excluida, `-3` valoración global, `-5` Comentario General, `-6` Por descubrir. Si algún chequeo no es 1, la nota se limita a 6.
**Tabla de aspectos final:** `resultado.topicos` (una fila por aspecto con `p_media` / `p_mediana`); las filas especiales se reportan pero no se priorizan.

## Paso 5 — Encoder (solo `modo_aspectos = "bertopic"`)

Se crea **una** instancia y se comparte entre grid, modelo final y encuestas: el modelo se carga una
vez y los embeddings se cachean por hash completo del contenido (el MVP 2 hasheaba una muestra de
500 textos).

La celda compara pares **originales** contra sus **textos de aspecto**: la tercera fila muestra por
qué la neutralización es correcta para agrupar aspectos — y por qué el sentimiento necesita su
propia rama.
> En modo `u4` este paso (y los Pasos 6, 7, 8 y 8b) se omiten solos: ver los Pasos U1–U7.


In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

codificador = ft.EncoderSentenceTransformers(cfg_base)

pares = [
    ("resuelve todas las dudas", "aclara las inquietudes de los estudiantes"),
    ("es una excelente profesora", "es un excelente profesor"),
    ("explica muy bien", "no explica nada bien"),
    ("llega puntual a clase", "el material de lectura es pertinente"),
]
filas = []
for a, b in pares:
    va, vb = codificador([a, b])
    aa, _ = neutralizador.neutralizar(a, ft.normalizar_clean(a, recursos))
    ab, _ = neutralizador.neutralizar(b, ft.normalizar_clean(b, recursos))
    sim_asp = float(np.dot(*codificador([aa, ab]))) if aa and ab else np.nan
    filas.append({"A": a, "B": b, "sim_original": round(float(va @ vb), 3),
                  "aspecto_A": aa or "(global)", "aspecto_B": ab or "(global)",
                  "sim_aspecto": round(sim_asp, 3)})
pd.DataFrame(filas)

#### 📖 Cómo leer esta salida

**Demostración con frases inventadas**, no resultados. Justifica por qué hay dos ramas: los tópicos se calculan sobre el texto neutralizado y el sentimiento sobre el original.
Fila 3: *"explica muy bien"* y *"no explica nada bien"* se parecen poco como frases (0,31), pero son **el mismo aspecto**: al neutralizar ambas quedan en *"explica"* (1,00). `sim` va de −1 a 1; más alto = más parecido.

## Paso 6 — Búsqueda de configuración con puertas duras (H6, H7) (solo `modo_aspectos = "bertopic"`)

1. Cada combinación se evalúa **tras fusión y reasignación reales**, con las mismas funciones que
   usará el modelo final. La cifra de la búsqueda es la cifra del modelo.
2. **Puertas duras** (`CriteriosNegocio`): todas deben cumplirse. El puntaje solo ordena lo que ya
   cumple y es **monótono**: más silueta y menos concentración siempre suben el puntaje.
3. **Frente de Pareto** en (silueta ↑, tópico mayor ↓, outliers ↓).
4. **Estabilidad**: ARI entre la partición y la de otras semillas UMAP; nunca se elige una
   configuración sin ARI medido.
5. Particiones idénticas se marcan `duplicada`; se barren `eom` **y** `leaf`.
6. **Fusión iterativa:** los tópicos redundantes (coseno ≥ `umbral_fusion`, 0,85) se fusionan
   hasta que ningún par resultante lo supere (H24).

**Tiempo:** `"rapida"` = 2 proyecciones UMAP + 32 HDBSCAN + 4 UMAP de estabilidad. `"completa"` = 5 + 180 + 6.


In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

criterios = ft.CriteriosNegocio(
    n_topicos_max=PERFIL["n_topicos_max"], pct_outliers_max=PERFIL["pct_outliers_max"],
    peso_silueta=PERFIL["peso_silueta"], peso_estabilidad=PERFIL["peso_estabilidad"],
)

pd.Series(vars(criterios)).to_frame("valor")

#### 📖 Cómo leer esta salida

**Qué muestra:** las **puertas duras** que debe cumplir una configuración (rango de tópicos, % outliers, tamaño del tópico mayor y de los dos mayores, silueta y estabilidad) y los **pesos** del puntaje que ordena a las que cumplen.
Son decisiones de negocio, no resultados. **Perilla:** `CriteriosNegocio(...)` en esta celda.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

textos_aspecto = aspecto["texto_aspecto"].tolist()
cfg_focal = ft.replace(cfg_base, nombre=FOCAL)

grid = ft.GridSearchAspectos(cfg_focal, recursos, codificador, criterios).preparar(textos_aspecto)
grid.modelador.clave_stopwords = encuesta.clave_stopwords
tabla_grid = grid.ejecutar(REJILLA)

#### 📖 Cómo leer esta salida

**Qué muestra:** el grid search, la parte larga (~15 min). Cada combinación prueba un modo de agrupar:
- `nn` (vecinos de UMAP): cuánto mira el vecindario. Más alto = grupos más amplios.
- `mts` (tamaño mínimo de tópico): menos que eso no es tópico.
- `ms` y `eom/leaf`: qué tan estricto es HDBSCAN. `leaf` da más tópicos y más finos.

A la derecha, lo que produjo: tópicos, % outliers, % del mayor (`top1`) y silueta (separación; más alto = mejor).
Las líneas *"estabilidad fila X: ARI"* repiten la agrupación con otras semillas; ARI 1 = sale idéntica. La última línea dice cuántas combinaciones cumplieron las puertas.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

COLS = ["umap_n_neighbors", "min_topic_size", "min_samples", "metodo", "n_topicos",
        "pct_outliers_hdbscan", "pct_outliers", "pct_top1", "pct_top2", "silueta",
        "ari_medio", "pareto", "puntaje", "cumple", "puertas_fallidas", "duplicada"]
print(f"Combinaciones que cumplen: {int(tabla_grid['cumple'].sum())} de {len(tabla_grid)}")
tabla_grid[COLS].head(12)

#### 📖 Cómo leer esta salida

**Qué muestra:** las combinaciones que cumplen, ordenadas por `puntaje`. **La fila 0 es candidata**, pero solo se elige si además es estable.
`pct_outliers_hdbscan` = sin tópico **antes** de fusionar y reasignar. `pct_outliers` = sin tópico **después**. Una diferencia grande indica que la reasignación hace mucho trabajo (ver Paso 8).
`ari_medio = NaN` quiere decir *no medido*, no *malo*: medir cuesta minutos, y solo se mide a las candidatas.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

tabla_grid["pareto"].value_counts(dropna=False)

#### 📖 Cómo leer esta salida

Cuántas combinaciones son **Pareto** (ninguna otra es mejor en todo a la vez). Es un control, no un resultado.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

from scipy.stats import spearmanr

unicas = tabla_grid[~tabla_grid["duplicada"]]
rho_sil, _ = spearmanr(unicas["puntaje"], unicas["silueta"])
rho_con, _ = spearmanr(unicas["puntaje"], unicas["pct_top1"])
print(f"Spearman puntaje~silueta: {rho_sil:+.2f}  (MVP 2: -0.51; debe ser > 0)")
print(f"Spearman puntaje~top1:    {rho_con:+.2f}  (MVP 2: +0.32; debe ser < 0)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for cumple, color, marca in [(False, GRIS, "o"), (True, VERDE, "*")]:
    sub = unicas[unicas["cumple"] == cumple]
    ax[0].scatter(sub["pct_top1"], sub["silueta"], c=color, marker=marca,
                  s=160 if cumple else 45, edgecolor="black", lw=0.4,
                  label="cumple" if cumple else "no cumple")
ax[0].axvline(criterios.pct_topico_mayor_max, ls="--", c=ROJO, lw=1)
ax[0].axhline(criterios.silueta_min, ls="--", c=ROJO, lw=1)
ax[0].set_xlabel("% del tópico mayor"); ax[0].set_ylabel("silueta (UMAP)")
ax[0].set_title("Separación vs. concentración"); ax[0].legend(fontsize=8)

for metodo, color in [("eom", AZUL), ("leaf", NARANJA)]:
    sub = unicas[unicas["metodo"] == metodo]
    ax[1].scatter(sub["n_topicos"], sub["pct_outliers"], c=color, alpha=0.8, label=metodo)
ax[1].axhline(criterios.pct_outliers_max, ls="--", c=ROJO, lw=1)
ax[1].axvspan(criterios.n_topicos_min, criterios.n_topicos_max, color=VERDE, alpha=0.1)
ax[1].set_xlabel("nº de tópicos"); ax[1].set_ylabel("% outliers tras reasignación")
ax[1].set_title("eom vs. leaf"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

#### 📖 Cómo leer esta salida

**Control del puntaje, no resultado.** El puntaje debe premiar la separación (signo `+` con silueta) y castigar la concentración (signo `−` con top1). Con esos signos, el ranking no favorece un tópico gigante, que era el error del MVP 2 (H6).

**Selección.** `SELECCION = None` elige la **primera fila que cumple también la puerta de
estabilidad** (ARI ≥ 0,70). Si ninguna la alcanza, con `PERMITIR_NO_CUMPLE = True` se usa la más
estable de las medidas y el log lo advierte (G2 fallará). Cada medición recalcula UMAP con otras
semillas: puede sumar varios minutos. Si ninguna fila cumple las puertas, `mejor_config` **falla**
en vez de devolver la mejor de las malas.


In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

# SELECCION y PERMITIR_NO_CUMPLE vienen de la celda CONFIG (Paso 0a).

try:
    cfg_final = grid.mejor_config(indice=SELECCION, permitir_no_cumple=PERMITIR_NO_CUMPLE)
except ft.SinConfiguracionValida as exc:
    print("⚠", exc)
    print("\nFilas más cercanas a cumplir:")
    display(tabla_grid[~tabla_grid["duplicada"]][COLS].head(5))
    raise

#### 📖 Cómo leer esta salida

**La línea a citar:** qué configuración se eligió y su ARI.
**Alarma:** la advertencia *"Ninguna configuración alcanzó ARI ≥ 0,70"*: se usará la más estable, pero la G2 fallará. Eso dice algo sobre esta encuesta (la agrupación cambia según la semilla), no es un error de código.

## Paso 7 — Modelo final y conciliación (H1, H2, H7) (solo `modo_aspectos = "bertopic"`)

`procesar_encuesta` reutiliza los embeddings **y la proyección UMAP** del grid, de modo que la
partición es idéntica a la fila elegida. La conciliación lo verifica con la huella de partición y
comprueba que todos los denominadores cuadran: **todas las filas `chequeo` deben valer 1**.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

clasificador = ft.ClasificadorRobertuito(cfg_final)

resultado = ft.procesar_encuesta(
    encuesta, cfg_final, recursos,
    codificador=codificador, clasificador=clasificador,
    unidades=(documentos, clausulas_base), grid=grid,
)
print("Modo de aspectos:", resultado.modo, "| sentimiento calibrado:",
      resultado.metricas["sentimiento_calibrado"])
resultado.conciliacion

#### 📖 Cómo leer esta salida

**Tres cosas que leer en el log:**
1. `partición:` es la historia del modelo. HDBSCAN encuentra N grupos y deja X % sin grupo → se fusionan los redundantes → se reasignan outliers por cercanía → quedan M tópicos. Es la frase para explicar cómo salieron los tópicos.
2. `etiquetas corregidas (calibrador)` = % de cláusulas cuyo sentimiento cambió respecto a RoBERTuito solo.
3. `[OK]` = el resumen de toda la encuesta en una línea.

**Tabla de conciliación:** cada `chequeo` debe valer **1**: los conteos cuadran de principio a fin. Si alguno no es 1, la nota se limita a 6.
Códigos: `-1` outlier, `-2` excluida, `-3` valoración global (el `-4`, alerta, se retiró: las alertas son de la Etapa 1).

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

pd.Series({k: v for k, v in resultado.metricas.items()}).to_frame("valor")

#### 📖 Cómo leer esta salida

**Glosario de métricas:**
- `cobertura_aspectual`: % de cláusulas con aspecto que terminaron en un tópico. `cobertura_directa`: solo las que HDBSCAN asignó sin reasignación.
- `pct_top1/2/3`: peso de los tópicos más grandes. `gini`: desigualdad de tamaños (0 = todos iguales, 1 = uno se lleva todo).
- `silueta`: separación entre tópicos. `max_coseno`: parecido del par de tópicos más parecido (G7 exige < 0,90). `diversidad`: % de palabras no repetidas entre etiquetas.
- `ari_medio`: estabilidad. `pct_neu_ajustado`: % de cláusulas con sentimiento corregido. `confianza_media`: seguridad promedio del modelo de sentimiento.

## Paso 8 — Aspectos descubiertos (solo `modo_aspectos = "bertopic"`)

`clausulas` cuenta cláusulas; `documentos` cuenta respuestas distintas que mencionan el aspecto
(una respuesta puede tocar varios). `cohesion` es el coseno medio al centroide: los tópicos con
cohesión baja son los primeros a revisar en la muestra de pureza.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

topicos = resultado.topicos
topicos[["topico", "etiqueta", "clausulas", "documentos", "cohesion",
         "crecimiento_reasignacion_pct", "ejemplos"]]

#### 📖 Cómo leer esta salida

**La tabla central del proyecto.** Una fila por tópico, del más grande al más pequeño.
- `etiqueta`: palabras **distintivas** del tópico (c-TF-IDF), no las más frecuentes. Antes de ponerle nombre, lea los `ejemplos`: la etiqueta puede engañar.
- `cohesion`: qué tan parecidas son sus cláusulas al centro del tópico. Más alto = más homogéneo.
- `crecimiento_reasignacion_pct`: cuánto creció el tópico con outliers reasignados. El tope es 35 % (`tope_crecimiento`). Si muchos tópicos están en 35 %, la reasignación está saturada y buena parte de cada tópico entró por cercanía, no por HDBSCAN.

**Perillas:** `tope_crecimiento` y `umbral_outliers` (reasignación), `umbral_fusion` (fusión).

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

fig, ax = plt.subplots(1, 2, figsize=(15, max(4, 0.32 * len(topicos))))
orden = topicos.sort_values("clausulas")
ax[0].barh(orden["etiqueta"].str[:55], orden["clausulas"], color=AZUL)
ax[0].set_title("Cláusulas por aspecto")
colores = [ROJO if c < topicos["cohesion"].quantile(0.2) else VERDE for c in orden["cohesion"]]
ax[1].barh(orden["etiqueta"].str[:55], orden["cohesion"], color=colores)
ax[1].set_title("Cohesión (rojo = quintil inferior: revisar)")
ax[1].set_yticklabels([])
plt.tight_layout(); plt.show()

arr = resultado.clausulas["topico"]
print(f"Cobertura aspectual: {resultado.metricas['cobertura_aspectual']}% | "
      f"top1 {resultado.metricas['pct_top1']}% | top2 {resultado.metricas['pct_top2']}% | "
      f"Gini {resultado.metricas['gini']} | fusiones {resultado.metricas['pares_fusionados']}")

#### 📖 Cómo leer esta salida

**Izquierda:** tamaño de cada aspecto. **Derecha:** cohesión; en rojo, el 20 % menos cohesionado (candidatos a revisar).
La pregunta clave: ¿los 2 o 3 tópicos más grandes son aspectos concretos o categorías amplias? Si son amplias, buena parte del corpus queda en tópicos poco accionables, aunque todas las puertas pasen.

In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

if resultado.modelador is not None and resultado.modelador.modelo is not None:
    display(resultado.modelador.modelo.visualize_barchart(
        top_n_topics=min(12, len(topicos)), n_words=8, height=260))

## Paso 8b — Ayuda de nombrado y tópicos a revisar (solo `modo_aspectos = "bertopic"`)

`aspecto_sugerido` es el aspecto de la taxonomía más afín al centroide de cada tópico, con las
descripciones **centradas** (se resta su componente común para que ninguna atraiga a todos). Es una
ayuda para ponerle nombre, **no una puerta**: la validez del tópico la decide su pureza humana.

`revisar` marca los tópicos con cohesión por debajo de la media − 1σ. Junto con la tasa de palabras
sueltas alimenta la columna `confiable` del ABSA (Paso 11). Con `podar_cohesion=True` en `Config` se
disuelven en outliers.


In [ ]:
if MODO_ASPECTOS != "bertopic":
    print("Paso omitido: modo_aspectos = 'u4' (los aspectos los asigna U4; ver los Pasos U1–U7). "
          "Para usar UMAP/HDBSCAN en toda la pregunta ponga modo_aspectos='bertopic' en PERFILES.")
    raise StopExecution

resultado.topicos[["topico", "etiqueta", "aspecto_sugerido", "afinidad_aspecto",
                   "cohesion", "revisar", "clausulas"]]

#### 📖 Cómo leer esta salida

**Ayuda para nombrar, no una clasificación.**
- `aspecto_sugerido`: el código de la taxonomía más cercano al tópico. `afinidad_aspecto` va de 0 a 1. **Por debajo de ~0,30, ignore la sugerencia** (ej.: *dinámicas* → CLARIDAD con 0,19).
- `revisar = True`: cohesión por debajo de media − 1σ. Es un umbral relativo: siempre marca a los menos cohesionados, aunque todos sean buenos.

Para comunicar resultados, nombre cada tópico por su contenido y agrúpelos bajo la taxonomía **usted**: varios tópicos pueden ser el mismo aspecto (p. ej., tres de *conocimiento*).

## Paso 9 — Sentimiento (H8)

Se conservan **dos** etiquetas:

- `sentimiento_modelo`: el argmax de RoBERTuito.
- `sentimiento`: reinterpreta NEU como POS/NEG cuando `|p_pos − p_neg| ≥ margen_neu`. En la
  auditoría, NEU tenía confianza media 0,63 y absorbía elogios como "Es inteligente".

`polaridad = p_pos − p_neg` es la medida continua que se valida contra la nota.

In [ ]:
cl = resultado.clausulas.dropna(subset=["sentimiento"])

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
comparacion = pd.DataFrame({
    "modelo": cl["sentimiento_modelo"].value_counts(normalize=True),
    "ajustado": cl["sentimiento"].value_counts(normalize=True),
}).reindex(["POS", "NEU", "NEG"]) * 100
comparacion.T.plot.bar(stacked=True, ax=ax[0], color=[COLOR_SENT[s] for s in comparacion.index])
ax[0].set_title("Distribución (%)"); ax[0].tick_params(axis="x", rotation=0)

for s in ["POS", "NEU", "NEG"]:
    ax[1].hist(cl.loc[cl["sentimiento_modelo"] == s, "confianza"], bins=25,
               histtype="step", lw=1.8, color=COLOR_SENT[s], label=s)
ax[1].set_title("Confianza por etiqueta del modelo"); ax[1].legend()

ax[2].hist(cl["polaridad"], bins=40, color=MORADO)
ax[2].set_title("Polaridad continua (p_pos − p_neg)")
plt.tight_layout(); plt.show()

print("NEG por presencia de negador:")
display(pd.crosstab(cl["tiene_negador"], cl["sentimiento"], normalize="index").round(3))

#### 📖 Cómo leer esta salida

**Tres gráficas:**
1. Distribución POS/NEU/NEG del modelo contra la ajustada: muestra cuánto movió el calibrador.
2. Qué tan seguro está el modelo por etiqueta.
3. Polaridad continua (p_pos − p_neg, de −1 a 1).

**Tabla del negador:** con negador, el % NEG debe subir; si no sube, el modelo no está leyendo las negaciones. Que haya POS con negador es normal (*"nunca falta"*).
Según la pregunta: en **fortalezas** (obs19, obs24) se espera mayoría POS; en **oportunidades de mejora** (obs20, obs25) mayoría NEG o NEU (sugerencias); en **obs23** (percepción del progreso de los estudiantes) el texto es en buena parte **descriptivo**: se espera una proporción alta de NEU. No es un sesgo del modelo, es lo que se preguntó.

In [ ]:
ajustadas = cl[cl["neu_ajustado"] == True]  # noqa: E712
print(f"NEU reinterpretadas: {len(ajustadas):,} ({100*len(ajustadas)/len(cl):.1f}%). Muestra:")
ajustadas.sample(min(15, len(ajustadas)), random_state=2)[
    ["clausula", "p_pos", "p_neu", "p_neg", "sentimiento"]] if len(ajustadas) else None

#### 📖 Cómo leer esta salida

**Revisión humana rápida del calibrador:** cláusulas que RoBERTuito dejaba en NEU y el calibrador reinterpretó. Cuente en cuántas de las 15 está de acuerdo con la etiqueta final: es una estimación instantánea de su precisión.
Error típico en preguntas de fortaleza: una **sugerencia** (*"debería…"*) marcada como POS.

## Paso 10 — Validación externa contra la nota

Dos niveles: **cláusula** (cada una con la nota de su documento) y **documento** (polaridad media),
que es donde la nota realmente vive. `auc_nota_baja` mide si la polaridad detecta a quien calificó
bajo. `monotonia_deciles` = 1 significa que la polaridad sube en cada decil de nota.

**Umbrales por tipo de pregunta (fijados a priori):** FORTALEZA ρ ≥ 0,25 y AUC ≥ 0,65;
MEJORA ρ ≥ 0,35 y AUC ≥ 0,70.

**Advertencia de diseño:** Una pregunta de fortalezas pregunta por *aspectos a resaltar*, así que el texto es positivo casi
por construcción y la nota se concentra arriba. Una correlación moderada aquí es esperable; en
preguntas de oportunidades de mejora debe ser más alta.

In [ ]:
val = resultado.validacion
display(pd.Series({k: v for k, v in val.items() if not isinstance(v, pd.DataFrame)}).to_frame("valor"))
if val.get("disponible"):
    display(val["por_polaridad"])

#### 📖 Cómo leer esta salida

**Si el texto lo escribe el docente (obs23, obs24, obs25):** la nota la ponen los estudiantes, así que esta validación queda `NO_APLICA` (H14). Es lo esperado; la prueba del sentimiento aquí es la G11.

**Validación externa:** ¿el sentimiento del texto va en la misma dirección que la nota numérica que puso el mismo estudiante?
- `nota_media_POS > NEU > NEG` en ese orden = la dirección es correcta.
- `monotonia_deciles` cerca de 1 = a más nota, más polaridad, de forma consistente.
- `spearman_documento` (G8) y `auc_nota_baja` (G9) miden la **fuerza**. En preguntas de fortaleza la nota está pegada al techo (mediana ≈ 4,9 de 5): hay poca variación que predecir, y esa es la razón principal de que estas puertas fallen.
- Criterio del proyecto: la nota es un indicio; la prueba del sentimiento es la G11 (anotación humana). No se mueven umbrales después de ver los resultados.

In [ ]:
if val.get("disponible"):
    deciles = val["deciles"]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(deciles["nota_media"], deciles["polaridad_media"], "o-", color=AZUL, lw=2)
    for d, fila in deciles.iterrows():
        ax.annotate(str(d), (fila["nota_media"], fila["polaridad_media"]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("nota media del decil"); ax.set_ylabel("polaridad media")
    ax.set_title(f"Polaridad por decil de nota (monotonía {val['monotonia_deciles']:+.2f})")
    plt.tight_layout(); plt.show()

#### 📖 Cómo leer esta salida

Cada punto es un decil de nota (del 10 % más bajo al más alto). **Normal:** una línea que sube. Una línea plana o en zigzag significaría que el sentimiento no sigue la nota.

## Paso 11 — ABSA: matriz de priorización (H9)

`prioridad = Wilson_inferior(%NEG) × log(1 + volumen)`. Un aspecto sube por negatividad **cierta**,
no por tamaño. Las filas especiales (valoración global, sin tópico) se reportan pero no se
priorizan, y las celdas con menos de `n_min_celda` cláusulas quedan sin prioridad.

**Tópicos no confiables.** Un tópico con cohesión baja (`revisar`) **y** con al menos el doble de
textos de aspecto de una sola palabra que el corpus queda `confiable = False`: no se esconde, pero
baja al final de `rango_prioridad`.


In [ ]:
absa = resultado.absa
absa[["etiqueta", "total_clausulas", "documentos", "pct_positivo", "pct_neutro", "pct_negativo",
      "neg_ic_inf", "neg_ic_sup", "polaridad_media", "n_suficiente", "prioridad",
      "veces_base_suelta", "confiable", "rango_prioridad"]]

#### 📖 Cómo leer esta salida

**La matriz ABSA** (aspecto × sentimiento). Una fila por tópico:
- `%` POS/NEU/NEG.
- `neg_ic_inf/sup`: intervalo de confianza del 95 % del % negativo (Wilson). Con pocos datos, el intervalo es ancho.
- `prioridad` = límite inferior del intervalo × log(volumen): premia un % negativo alto **y seguro** en un tópico con volumen.
- `confiable = False`: cohesión baja **y** 2 o más veces la tasa de palabras sueltas del corpus. Queda al final del ranking (`rango_prioridad`) aunque su prioridad sea alta.
- Filas especiales: `-3` valoración global, `-1` sin tópico; en modo u4 también `-5` «Comentario General», `-6` «Por descubrir» y las clases sin aspecto de U4 (p. ej. «Opinión general»). En modo u4 cada fila es un **aspecto fijo** de U4 (no un tópico descubierto).

**Cómo leerla según la pregunta:** en **fortalezas** (obs19, obs24) el mensaje principal es qué se valora: ordene por volumen y lea los POS; los NEG son excepciones. En **oportunidades de mejora** (obs20, obs25) la priorización por negativo es el resultado principal. En **obs23** el docente describe el avance de sus estudiantes: ordene por volumen y lea los tres sentimientos; los NEG señalan dificultades de aprendizaje que reporta el propio docente, no quejas sobre él.

In [ ]:
asp = absa[absa["es_aspecto"] & absa["n_suficiente"]]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.errorbar(asp["total_clausulas"], asp["pct_negativo"],
            yerr=[asp["pct_negativo"] - asp["neg_ic_inf"], asp["neg_ic_sup"] - asp["pct_negativo"]],
            fmt="none", ecolor=GRIS, lw=1)
sc = ax.scatter(asp["total_clausulas"], asp["pct_negativo"], s=60 + 900 * asp["prioridad"] / asp["prioridad"].max(),
                c=asp["polaridad_media"], cmap="RdYlGn", edgecolor="black", lw=0.6, zorder=3)
for t, fila in asp.iterrows():
    ax.annotate((f"T{t}" if MODO_ASPECTOS == "bertopic" else str(fila["etiqueta"])[:16]) + ("" if fila["confiable"] else "*"), (fila["total_clausulas"], fila["pct_negativo"]), fontsize=8,
                ha="center", va="center", weight="bold", zorder=4)
ax.set_xscale("log")
ax.axhline(asp["pct_negativo"].median(), ls="--", c="gray", lw=1)
ax.set_xlabel("volumen (cláusulas, escala log)"); ax.set_ylabel("% negativo (barras: IC Wilson 95%)")
ax.set_title(f"{FOCAL} — priorización de aspectos (tamaño = prioridad; * = no confiable)")
plt.colorbar(sc, ax=ax, label="polaridad media")
plt.tight_layout(); plt.show()

#### 📖 Cómo leer esta salida

**Mapa de priorización.** Eje x = volumen (escala log). Eje y = % negativo con su intervalo. Tamaño del punto = prioridad. Color = polaridad media. `*` = no confiable.
Arriba a la derecha = mucho volumen y mucho negativo = atender primero. La línea punteada es la mediana de % negativo.

In [ ]:
for t in asp.sort_values("rango_prioridad").head(5).index:
    fila = absa.loc[t]
    sub = resultado.clausulas[resultado.clausulas["topico"] == t]
    print(f"\n▸ {fila['etiqueta']}  |  {int(fila['total_clausulas']):,} cláusulas · "
          f"NEG {fila['pct_negativo']}% (IC {fila['neg_ic_inf']}–{fila['neg_ic_sup']}) · "
          f"prioridad {fila['prioridad']:.3f} · rango {fila['rango_prioridad']}")
    for texto in sub[sub["sentimiento"] == "NEG"].sort_values("p_neg", ascending=False)["clausula"].head(3):
        print(f"   − {texto[:115]}")
    for texto in sub[sub["sentimiento"] == "POS"].sort_values("p_pos", ascending=False)["clausula"].head(2):
        print(f"   + {texto[:115]}")

#### 📖 Cómo leer esta salida

**Top 5 para leer en voz alta:** cada aspecto con 3 ejemplos negativos (−) y 2 positivos (+). Es la evidencia cualitativa para presentar.
Antes de comunicar un aspecto, lea sus ejemplos: si los negativos no hablan del aspecto de la etiqueta (p. ej., quejas generales del docente), el tópico es más amplio de lo que su nombre sugiere.

## Paso 12 — Scorecard y exportación

La nota sale de puertas numéricas, no de opinión. Dos topes explícitos: sin integridad (G1) la nota
no supera 6; con validación humana pendiente (G10 pureza, G11 F1) no supera **8**. Por eso superar
8 exige completar el Paso 13.

La exportación escribe **todo desde el mismo estado en memoria** y relee parquet y CSV para
verificar que coinciden (H1). Las métricas incluyen `cobertura_directa` (cláusulas que HDBSCAN
asignó sin reasignación) junto a `cobertura_aspectual`; la puerta G3 usa la cobertura total.

**Modo u4 (ft-r4.15):** las puertas de silueta, estabilidad (ARI), outliers, concentración, redundancia y pureza de tópicos quedan `NO_APLICA` (con el motivo escrito en `umbral`) porque no hay partición UMAP/HDBSCAN del corpus. Se añaden seis puertas **informativas** `PENDIENTE` (sin umbral definido por el equipo): acierto y F1 macro en hold-out, cobertura, % «Comentario General», % de aspecto de la lista (excluye «Por descubrir») y % «Por descubrir». La fórmula de la nota **no cambia** (solo cuentan las puertas aplicables y medidas), así que la nota no se infla.


In [ ]:
tarjeta = pd.DataFrame(resultado.scorecard["puertas"])
def color(estado):
    return {"PASA": f"background-color: {VERDE}33", "FALLA": f"background-color: {ROJO}33",
            "NO_APLICA": f"background-color: {GRIS}55", "PENDIENTE": f"background-color: {NARANJA}33"}.get(estado, "")
display(tarjeta.style.map(color, subset=["estado"]))
print(f"NOTA: {resultado.scorecard['nota']} | topes: {resultado.scorecard['topes_aplicados']}")

#### 📖 Cómo leer esta salida

**Scorecard:** una puerta por fila con valor, umbral y estado.
**Nota** = % del **peso** de las puertas evaluadas que pasan, en escala 0–10. Pesos: G3 y G10 valen 1,5; G6 y G7 valen 0,5; el resto 1. Las puertas `PENDIENTE` no cuentan.
**Topes:** mientras falte la auditoría humana (G10/G11), la nota no pasa de 8.
**Qué significa:** G1–G7 dicen que la partición está bien *construida*. **No** dicen que los tópicos sean *útiles*: eso lo dice G10 (pureza humana).

In [ ]:
CARPETA = ft.exportar_resultado(resultado)
print("Artefactos de la corrida:")
for archivo in sorted(CARPETA.iterdir()):
    print(f"   {archivo.name:<22} {archivo.stat().st_size/1024:>9.1f} KB")

#### 📖 Cómo leer esta salida

**Archivos de la corrida** (carpeta con fecha y huella; ninguna corrida pisa a otra). **Solo traen respuestas procesadas** (`es_no_respuesta = False` y `tiene_alerta = False`): ningún archivo exportado muestra datos de no-respuestas ni de respuestas con alerta (la única excepción será `respuestas.csv` del NLP, que lleva todas las filas con sus dos banderas). La celda siguiente lo comprueba:
- `clausulas.csv/.parquet`: una fila por cláusula con tópico, sentimiento y probabilidades. La base para cualquier análisis propio.
- `topicos.csv` y `absa.csv`: las tablas de los Pasos 8 (o U7) y 11. En modo u4 `topicos.csv` es la **tabla de aspectos**.
- `documentos.csv`: una fila por fila del corpus; en modo u4 añade `respuesta_completa`, `n_palabras_respuesta` y `respuesta_corta`. `doc_id` es la llave de unión con `clausulas.csv`.
- Modo u4, además: columnas `aspecto`, `aspecto_u4`, `p_aspecto_u4`, `p_u4_<aspecto>`, `respuesta_corta`, `u4_visto_en_entrenamiento`, `asignacion`, `es_residual`, `topico_residual` (vacío si la cláusula no es residual; -1 = residual sin grupo), `etiqueta_residual`, `u4_baja_confianza` y `modelo_u4_id` en `clausulas.csv`; `topicos_residuales.csv` (y hoja en `auditoria.xlsx`); y en `manifest.json` el modo, el modelo U4, los parámetros y métricas del residual y la huella del archivo auditado usado.
- El **modelo U4** no va en esta carpeta: queda en `RAIZ/models/modelos/u4_aspectos/<clave>/<run_id>/` (con su `entrenamiento_u4.xlsx`).
- `auditoria.xlsx`: lo que se diligencia a mano para G10/G11.
- `scorecard.json`, `grid.csv`, `manifest.json`: trazabilidad (qué versión, qué configuración, qué nota).

In [ ]:
import re

_docs = pd.read_csv(CARPETA / "documentos.csv")
_clau = pd.read_csv(CARPETA / "clausulas.csv")

# Una fila por cláusula, con los datos de su documento al lado; los documentos sin cláusulas (vacíos, ruido) quedan con las columnas de cláusula vacías.
_unif = _docs.merge(_clau, on="doc_id", how="left", suffixes=("_doc", ""), validate="one_to_many")
assert len(_unif) == len(_clau) + int((~_docs["doc_id"].isin(_clau["doc_id"])).sum()), "la unión cambió el número de filas"

# Excel rechaza caracteres de control: se eliminan solo en las columnas de texto
_ilegal = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
for _c in _unif.select_dtypes(include="object").columns:
    _unif[_c] = _unif[_c].map(lambda v: _ilegal.sub("", v) if isinstance(v, str) else v)

_ruta_xlsx = CARPETA / "clausulas_documentos_unificado.xlsx"
_unif.to_excel(_ruta_xlsx, index=False, sheet_name="unificado")
print(f"Guardado: {_ruta_xlsx} | filas: {len(_unif):,} | columnas: {_unif.shape[1]}")

# Comprobación: ningún archivo de la corrida trae respuesta_id de no-respuestas ni de respuestas con alerta (solo conteos).
from nlp_utilidades import salidas_contrato as sc

_resp_etapa1 = pd.read_csv(CARPETA_CORRIDA_ETAPA1 / "respuestas.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
_verif_archivos = sc.verificar_archivos_sin_excluidas(CARPETA, _resp_etapa1)   # falla con ErrorSalidasNLP si encuentra alguna
display(_verif_archivos)
print(f"✅ {len(_verif_archivos)} archivos/hojas revisados: ninguno trae respuestas excluidas "
      f"({len(sc.ids_respuestas_excluidas(_resp_etapa1)):,} respuestas excluidas en la corrida de la Etapa 1).")


## Paso 12b — Salidas del contrato de datos v0.2 y modelo del residual

**Solo modo `u4`.** Convierte lo que acaba de producir el pipeline en las **cuatro tablas del contrato** (`clausulas`, `auditoria_clasificacion`, `catalogo_aspectos`, `metadatos_nlp`; las alertas ya no son del NLP: `alertas.csv` lo publica la Etapa 1), las escribe en `RAIZ/data/processed/nlp/<id_corrida_nlp>/` (una carpeta por corrida: ninguna pisa a otra; utf-8-sig, coma, punto decimal, `True`/`False`, vacío para el dato ausente) y **valida** los criterios de aceptación del contrato (cobertura, IDs, integridad referencial, dominios, confianzas, auditoría, metadatos). Si una validación falla, la celda se detiene con un mensaje que lista todas las fallas (solo conteos).

Las columnas internas del modelo **no cambian**: `doc_id` sigue siendo la llave de unión interna y se traduce a `respuesta_id` aquí (`clausula_id = respuesta_id|orden`). Las salidas internas de los pasos anteriores (`data/processed/nlp/modelado/`) se conservan.

Las **decisiones abiertas** del equipo son los parámetros de `SALIDAS_CONTRATO` (celda CONFIG). «Comentario General» y «VALORACIÓN GLOBAL» se asignan a `OPINION_GENERAL` con `aspecto_origen = regla` (mapeo configurable en `salidas_contrato.MAPEO_ASIGNACION_DEFECTO`). Las respuestas con alerta no llegan al NLP (la ingesta solo toma `es_no_respuesta = False` y `tiene_alerta = False`); la validación comprueba que ninguna tenga cláusulas.

Además guarda el **UMAP y el HDBSCAN del residual** (ajustados con `prediction_data=True`, misma semilla y parámetros) junto al modelo U4, para asignar respuestas nuevas con `approximate_predict` sin reajustar.


In [ ]:
if MODO_ASPECTOS != "u4":
    print("Paso omitido: modo_aspectos = 'bertopic' (este paso solo aplica al modo 'u4').")
    raise StopExecution

from nlp_utilidades import salidas_contrato as sc
from nlp_utilidades.taxonomia import leer_taxonomia, version_taxonomia
from preparacion.configuracion import codigos_preguntas

_pregunta = ft.clave_logica(CLAVE_FOCAL)
_ruta_tax = DIR_LEXICOS / "taxonomia.xlsx"
_taxonomia = leer_taxonomia(_ruta_tax, codigos_preguntas(RUTA_CATALOGO))                 # valida la taxonomía

_params = sc.ParametrosSalidas(
    umbral_respaldo_hdbscan=cfg_final.umbral_residual_u4,
    regla_ajuste_sentimiento=sc.describir_regla_ajuste_sentimiento(cfg_final.margen_neu, bool(resultado.metricas["sentimiento_calibrado"])),
    **SALIDAS_CONTRATO,
)
_entrada = sc.entrada_desde_resultado(
    resultado, clases=modelo_u4.clases, columnas_p=ft.nombres_columnas_p(modelo_u4.clases),
    modelo_u4_id=resultado.u4["modelo_u4_id"], modelo_sentimiento=cfg_final.modelo_sentimiento,
)
salidas = sc.construir_salidas_contrato(
    [_entrada], taxonomia=_taxonomia,
    id_corrida_preparacion=ID_CORRIDA_PREPARACION, version_taxonomia=version_taxonomia(_ruta_tax),
    parametros=_params,
)
for _aviso in salidas.avisos:
    print("⚠️ ", _aviso)

CARPETA_NLP = RUTAS["nlp"] / salidas.id_corrida_nlp
sc.escribir_salidas_contrato(salidas, CARPETA_NLP)

# Validación sobre lo RELEÍDO como texto (dtype=str, utf-8-sig), contra respuestas.csv de la corrida procesada de la Etapa 1.
_respuestas = pd.read_csv(CARPETA_CORRIDA_ETAPA1 / "respuestas.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
sc.validar_salidas_nlp(
    sc.leer_salidas_contrato(CARPETA_NLP), _respuestas, id_corrida_preparacion=ID_CORRIDA_PREPARACION,
    preguntas_procesadas=[_pregunta], clases_por_pregunta={_pregunta: list(modelo_u4.clases)}, taxonomia=_taxonomia,
)
sc.verificar_archivos_sin_excluidas(CARPETA_NLP, _respuestas)   # ninguna tabla trae respuestas excluidas
print("✅ Las 4 tablas cumplen las validaciones del contrato y no traen respuestas excluidas.")
print("Carpeta:", CARPETA_NLP, "| id_corrida_nlp:", salidas.id_corrida_nlp)

_c = salidas.clausulas
display(pd.DataFrame({
    "clausulas": _c["aspecto_origen"].value_counts(),
    "%": (100 * _c["aspecto_origen"].value_counts() / len(_c)).round(2),
}).rename_axis("aspecto_origen"))
print(f"Cláusulas: {len(_c):,} | sin opinión (es_opinion = False): {(_c['es_opinion'] == 'False').sum():,} | "
      f"temas emergentes: {(salidas.catalogo['grupo_visual'] == 'Temas emergentes').sum()}")

# Estabilidad: UMAP + HDBSCAN del residual (prediction_data=True) junto al modelo U4, para asignar respuestas nuevas sin reajustar.
if etapas.residual_persistible is None:
    print("Residual: sin modelo persistible (no se ejecutó HDBSCAN o falló el reajuste; ver el log).")
else:
    print("Residual guardado en:", ft.guardar_modelo_residual(etapas.residual_persistible, modelo_u4.carpeta / "residual"))


#### 📖 Cómo leer esta salida

**Qué muestra:** que las cuatro tablas se construyeron, se escribieron y **pasaron todas las validaciones del contrato** (releídas como texto, como las leerá Power BI), la carpeta donde quedaron y el resumen: cláusulas por `aspecto_origen` (cantidad y %), cláusulas sin opinión y temas emergentes.
**Normal:** ✅ y porcentajes coherentes con los Pasos U3–U7 (el origen `regla` agrupa «Comentario General» y «Valoración global»; `sistema` = sin opinión y ruido).
**Alarma:** una `ErrorSalidasNLP` lista qué criterio falló (cobertura, IDs repetidos, aspecto fuera del catálogo, confianzas, etc.): corríjalo antes de entregar.


## Paso 13 — Auditoría humana (lleva la nota por encima de 8)

Abra `auditoria.xlsx` de la carpeta de la corrida y diligencie dos hojas:

1. **`muestra_pureza`** — 10 cláusulas **aleatorias** por tópico. En `pertenece_s_n` escriba `S` si
   la cláusula trata el aspecto del tópico, `N` si no. Son aleatorias a propósito: las
   representativas se ven bien por construcción.
2. **`muestra_manual`** — cláusulas estratificadas por etiqueta y confianza. En `etiqueta_humana`
   escriba `POS`, `NEU` o `NEG` **sin mirar** la columna `sentimiento`.

Guarde el Excel **con el mismo nombre**, cambie `AUDITORIA_DILIGENCIADA` a `True` y ejecute la celda.

*Modo u4:* `muestra_pureza` se sigue exportando, pero la puerta G10 (pureza de tópicos) queda `NO_APLICA`: la validez de los aspectos de U4 se mide con el hold-out del Paso U1. `muestra_manual` (sentimiento, G11) se diligencia igual.


In [ ]:
# AUDITORIA_DILIGENCIADA y RUTA_MUESTRA_ANTERIOR vienen de la celda CONFIG (Paso 0a).

if AUDITORIA_DILIGENCIADA:
    nueva = ft.recalificar(resultado, CARPETA, ruta_muestra_manual=RUTA_MUESTRA_ANTERIOR)
    display(pd.DataFrame(nueva["puertas"]).style.map(color, subset=["estado"]))
    print(f"NOTA RECALIFICADA: {nueva['nota']} | topes: {nueva['topes_aplicados']}")
    f1 = ft.evaluar_muestra_manual(CARPETA / "auditoria.xlsx")
    if f1.get("disponible"):
        display(f1["confusion"])

## Paso 14 — Otras encuestas (opcional)

Con `CORRER_RESTO = True` procesa **las demás encuestas de `MIS_ENCUESTAS`**.

- **Modo `bertopic`:** rejilla `REJILLA_RESTO` (`rapida`). Las columnas con menos de 300 cláusulas con aspecto solo reciben sentimiento; `permitir_no_cumple=True` hace que toda encuesta genere su `auditoria.xlsx`.
- **Modo `u4`:** para **cada** pregunta replica *entrenamiento + etiquetado*: entrena (o reutiliza con `REUTILIZAR_MODELO`, o solo carga con `SOLO_ETIQUETAR`) su modelo con **su** archivo auditado y **su** bloque `aspectos_u4` / `residual_hdbscan` de `PERFILES`, y luego corre el pipeline. Si a una pregunta le falta el archivo auditado (o columnas), se **informa con el error claro y se omite esa pregunta**; nunca se entrena con datos sin auditar.


In [ ]:
CORRER_RESTO = False
REJILLA_RESTO = "rapida"

if CORRER_RESTO and not [n for n in ASIGNADAS if n != FOCAL]:
    print("MIS_ENCUESTAS tiene una sola encuesta: no hay resto que procesar.")
elif CORRER_RESTO:
    resto = [n for n in ASIGNADAS if n != FOCAL]
    print("Se procesarán:", *resto, sep="\n  · ")
    if "clasificador" not in globals():
        clasificador = ft.ClasificadorRobertuito(cfg_base)
    if "codificador" not in globals():
        codificador = ft.EncoderSentenceTransformers(cfg_base)
    extra = {}
    if cfg_base.modo_aspectos == "u4":
        modelos_resto, omitidas = {}, {}
        for nombre in resto:
            clave = ft.clave_logica(nombre)
            if modo_de(clave) != "u4":
                continue                       # esta pregunta se procesa en modo bertopic (aspectos_u4.activo = False)
            try:
                modelos_resto[clave] = (
                    ft.modelo_u4_de_clave(RAIZ_MODELOS, clave) if SOLO_ETIQUETAR
                    else ft.entrenar_u4(clave, params_u4(clave), RAIZ_MODELOS, reutilizar=REUTILIZAR_MODELO, permitir_cpu=PERMITIR_CPU))
            except (ft.ErrorEtiquetasU4, FileNotFoundError) as exc:
                omitidas[nombre] = str(exc)
                print(f"\n⛔ {nombre}: se OMITE (no se entrena con datos sin auditar):\n   {exc}")
        resto = [n for n in resto if n not in omitidas]
        extra = dict(modelos_u4=modelos_resto,
                     config_por_encuesta={ft.clave_logica(n): campos_config(ft.clave_logica(n)) for n in resto})
        if omitidas:
            print("\nPreguntas omitidas por falta de datos auditados:", *omitidas, sep="\n  · ")
    if resto:
        resultados = ft.ejecutar_pipeline(
            cfg_base, raiz=RAIZ, dir_lexicos=DIR_LEXICOS, raiz_modelos=RAIZ_MODELOS, solo=resto, rejilla=REJILLA_RESTO, criterios=criterios,
            permitir_no_cumple=PERMITIR_NO_CUMPLE, codificador=codificador,
            clasificador=clasificador,
            carpeta_etapa1=RUTA_ETAPA1, **extra,
        )
        resultados[FOCAL] = resultado
        display(ft.resumen_comparativo(resultados))


## Paso 15 — Descargar todos los resultados

Comprime `data/processed/nlp/modelado` completo (una carpeta por encuesta y corrida) y lo descarga. Localmente el ZIP queda en esa misma carpeta (`outputs_full.zip`; no se incluye a sí mismo ni a otros ZIP); en Colab se guarda en `/content`.
El ZIP contiene las corridas de `modelado`; los **modelos U4** (cientos de MB) viven aparte en `RAIZ/models/modelos/u4_aspectos/` y no se comprimen aquí (con `GUARDAR_EN_DRIVE = True` ya están en Drive).


In [ ]:
zip_ruta = Path("/content/outputs_full.zip") if EN_COLAB else SALIDA / "outputs_full.zip"
archivo_zip = str(ft.comprimir_carpeta(SALIDA, zip_ruta))
print("ZIP:", archivo_zip)
for carpeta in sorted(SALIDA.glob("*/*/")):
    print(f"   {carpeta.parent.name:<45} {carpeta.name}")
if EN_COLAB:
    from google.colab import files
    files.download(archivo_zip)

#### 📖 Cómo leer esta salida

Lista **todas** las corridas guardadas en `data/processed/nlp/modelado`, incluidas las de sesiones anteriores y las de otros módulos. Identifique la suya por fecha y hora.

## Paso 16 — Recalificar con auditorías diligenciadas (sesión B)

Ejecute **solo el Paso 0** y luego estas celdas. No carga modelos.

1. Si la sesión se reinició y los resultados no están en Drive, la primera celda pide subir
   `outputs_full.zip`.
2. La segunda pide los `auditoria.xlsx` diligenciados. **Si ya están en Drive, cancele la subida.**
3. La tercera **ajusta el calibrador de sentimiento** con todas las anotaciones y lo guarda en
   `calibrador_sentimiento.json` (imprime por tipo de pregunta `f1_macro_cv`, el compartido y qué
   modelo se usa). Ejecútela solo si se decidió reentrenar el calibrador.
4. La cuarta recalcula las notas de las encuestas que elija (por número) y descarga el ZIP.

**Limitación conocida:** el tipo de pregunta se deduce del nombre de la columna. Una columna con
nombre no descriptivo queda como ABIERTA aunque sea de sugerencias.

**Modo u4:** la recalificación lee `manifest.json` y `auditoria.xlsx` de cada corrida (las métricas de U4, el hold-out y el residual van en el manifiesto), así que funciona igual para todas las preguntas sin cargar ni reentrenar modelos. La puerta G11 (F1 de sentimiento) es la que se actualiza; G10 sigue `NO_APLICA` y las puertas `U1`–`U6` siguen `PENDIENTE` hasta que el equipo fije umbrales.


#### 📖 Cómo leer esta salida

⚠️ **Salidas viejas:** si una celda de este paso muestra salida pero su contador está vacío `[ ]`, esa salida es de una sesión **anterior** y no corresponde a la corrida actual. Solo cuentan las celdas con número de ejecución.

In [ ]:
import shutil
import zipfile

from nlp_utilidades import modulos_nlp as ft

if not any(SALIDA.glob("*/*/manifest.json")):
    if not EN_COLAB:
        raise FileNotFoundError(f"No hay corridas en {SALIDA}")
    from google.colab import files
    print("No hay corridas en disco: suba outputs_full.zip")
    subido = files.upload()
    with zipfile.ZipFile(next(iter(subido))) as z:
        z.extractall(SALIDA)
print("Corridas disponibles:", len(list(SALIDA.glob("*/*/manifest.json"))))

In [ ]:
if EN_COLAB:
    from google.colab import files
    print("Suba los auditoria.xlsx diligenciados:")
    subidos = [Path(nombre) for nombre in files.upload()]
else:
    subidos = []  # rutas locales, p. ej. [Path("auditoria_P19.xlsx")]

carpetas = ft.instalar_auditorias(subidos, SALIDA)
print("Instaladas en:", *carpetas, sep="\n  · ")

In [ ]:
calibrador = ft.calibrar_sentimiento(SALIDA, RUTA_CALIBRADOR)
print(json.dumps(calibrador.metricas, ensure_ascii=False, indent=2))
print("\nGuardado en:", RUTA_CALIBRADOR)
print("Las notas de abajo NO usan aún el calibrador: se aplica en la próxima corrida (sesión A).")

In [ ]:
if EN_COLAB:
    from google.colab import files

ENCUESTAS_A_RECALIFICAR = None  # p. ej. [1, 3] (números) o "todas"
SOLO_ULTIMA_CORRIDA = True

_disponibles = sorted({
    json.loads(m.read_text(encoding="utf-8")).get("encuesta", m.parent.parent.name)
    for m in SALIDA.glob("*/*/manifest.json")
})


# CAMBIO: se numeran las encuestas y se pide el NÚMERO (o varios
# separados por coma, o "todas") -- más corto de escribir y sin riesgo de
# typo en el nombre. ENCUESTAS_A_RECALIFICAR sigue aceptando una lista de
# nombres exactos si se precarga a mano (p. ej. para uso desatendido).
if ENCUESTAS_A_RECALIFICAR is None:
    print("Encuestas con corridas disponibles para recalificar:")
    for _i, _clave in enumerate(_disponibles, start=1):
        print(f"  {_i}. {_clave}")
    _respuesta = input(
        "Escriba los NÚMEROS separados por coma, o 'todas': "
    ).strip()
    if _respuesta.lower() == "todas":
        solo = None
    else:
        _indices = [n.strip() for n in _respuesta.split(",") if n.strip()]
        if not all(n.isdigit() for n in _indices):
            raise ValueError(
                f"'{_respuesta}' no son sólo números separados por coma. "
                "Use los números de la lista de arriba, o 'todas'."
            )
        _indices = [int(n) for n in _indices]
        _fuera_de_rango = [i for i in _indices if i < 1 or i > len(_disponibles)]
        if _fuera_de_rango:
            raise ValueError(
                f"Números fuera de rango: {_fuera_de_rango}. "
                f"Debe ser entre 1 y {len(_disponibles)}."
            )
        solo = [_disponibles[i - 1] for i in _indices]
elif isinstance(ENCUESTAS_A_RECALIFICAR, str) and ENCUESTAS_A_RECALIFICAR.lower() == "todas":
    solo = None
else:
    solo = list(ENCUESTAS_A_RECALIFICAR)

resumen = ft.recalificar_todas(SALIDA, solo=solo, solo_ultima_corrida=SOLO_ULTIMA_CORRIDA)
display(resumen)


# CAMBIO: el detalle sale directo de `resumen` (encuesta + run_id
# ya calculados por recalificar_todas arriba), sin depender de si se subió
# algo en la celda de auditorias.
for _, _fila in resumen.iterrows():
    _carpeta = SALIDA / _fila["encuesta"] / _fila["run_id"]
    tarjeta = json.loads((_carpeta / "scorecard.json").read_text(encoding="utf-8"))
    print(f"\n{_fila['encuesta']} — NOTA {tarjeta['nota']} | topes: {tarjeta['topes_aplicados']}")
    display(pd.DataFrame(tarjeta["puertas"])[["clave", "valor", "umbral", "estado"]])

archivo_zip = str(ft.comprimir_carpeta(
    SALIDA, Path("/content/outputs_full_recalificado.zip") if EN_COLAB else SALIDA / "outputs_full_recalificado.zip"))
if EN_COLAB:
    files.download(archivo_zip)